# Data-analysis figures

Figure 1; Figure 2b–c; Figure 3a–b and inset; ED Figures 1–2, 4, 7, 8a–c.

Supply the inputs listed in `../DATA_MANIFEST.json`, then run Setup before the
relevant figure sections. Data access and directory settings are in `../README.md`.


In [ ]:
from pathlib import Path
import sys

_reproduction_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "src" / "reproduction_paths.py").is_file()),
    None,
)
if _reproduction_root is None:
    raise FileNotFoundError("Start Jupyter from the repository root or its notebooks folder.")
sys.path.insert(0, str(_reproduction_root / "src"))
from reproduction_paths import RAW_DATA_DIR, CACHE_DIR, MOVIE_DATA_DIR, CANN_RUNS_DIR, STYLE_PATH


## Setup


In [ ]:
from scipy.io import loadmat
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
plt.style.use(STYLE_PATH)
COLORS = {
    "FR": "#a82203",
    "CTR": "#208cc0",
    "compare": "#f1af3a",
    "Simulation": "#cf5e4e",
    "Theory": "#637b31",
    "Other": "#003967",
    "Purple": "#7B4FA3"
}

def mm_to_inch(mm):
    return mm / 25.4
def clean_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(length=2.5, width=0.6)
    return ax


In [ ]:
HUNGRY_DECONV = np.load((RAW_DATA_DIR / 'predictions_fullTrace_hungry.npy'), allow_pickle=True)

FOOD_RESTRICTED_HUNGRY = [1,2,3,6,7,9,11,12]
CONTROL_HUNGRY = [0,4,5,8,10,13]

AngStim_data = (RAW_DATA_DIR / 'metadata_deconv/stimAngle_hungry.mat')
ANG_STIM_DATA = loadmat(AngStim_data, simplify_cells= True)
HUNGRY_ANGLE = ANG_STIM_DATA['order_of_stim_arossAnimals']

SfStim_data = (RAW_DATA_DIR / 'metadata_deconv/stimSpatFreq_hungry.mat')
SF_STIM_DATA = loadmat(SfStim_data, simplify_cells= True)
HUNGRY_SF = SF_STIM_DATA['stimSpatFreq_arossAnimals']
SATED_DECONV = np.load((RAW_DATA_DIR / 'predictions_fullTrace_sated.npy'), allow_pickle=True)

AngStim_data = (RAW_DATA_DIR / 'metadata_deconv/stimAngle_sated.mat')
ANG_STIM_DATA = loadmat(AngStim_data, simplify_cells= True)
SATED_ANGLE = ANG_STIM_DATA['order_of_stim_arossAnimals']

SfStim_data = (RAW_DATA_DIR / 'metadata_deconv/stimSpatFreq_sated.mat')
SF_STIM_DATA = loadmat(SfStim_data, simplify_cells= True)
SATED_SF = SF_STIM_DATA['stimSpatFreq_arossAnimals']

FOOD_RESTRICTED_SATED = [1,2,3,6,7,8,11,12]
CONTROL_SATED         = [0,4,5,9,10,13]


In [ ]:
def resort_preprocessing(datum,angle_arr,sf_arr,animal):
    data = np.copy(datum[animal,:])
    neurons = data[0].shape[0]
    reshape_data = np.full((60,neurons,data[0].shape[1]), np.nan)
    for i in range(60):
        reshape_data[i,:,:] = data[i]

    reshape_data = reshape_data.reshape(60,neurons,12,120)
    reshape_data = np.transpose(reshape_data,(1,2,0,3))
    reshape_data = reshape_data[2:,:,:,:]

    trial_is_nan = np.isnan(reshape_data[0,1,:,0])
    max_trial = np.where(trial_is_nan)[0][0] if np.any(trial_is_nan) else trial_is_nan.size
    reshape_data = reshape_data[:,:,:max_trial,:]

    angles = np.copy(angle_arr[animal])
    for itrials in range(reshape_data.shape[2]):
        order = angles[:,itrials]-1
        reshape_data[:,:,itrials,:] = reshape_data[:,order,itrials,:]

    reshaped_data = []
    sfs = np.copy(sf_arr[animal])[:reshape_data.shape[2]]
    for experiment in range(1,6):
        mask = sfs == experiment
        reshaped_data.append(reshape_data[:,:,mask,:])

    max_trials = max([exp.shape[2] for exp in reshaped_data])
    for i in range(len(reshaped_data)):
        if reshaped_data[i].shape[2] < max_trials:
            padding = max_trials - reshaped_data[i].shape[2]
            reshaped_data[i] = np.pad(reshaped_data[i], ((0, 0),(0, 0),(0, padding),(0, 0)), mode='constant', constant_values=np.nan)

    reshaped_data = np.stack(reshaped_data,axis=2)
    dirty_data = np.copy(reshaped_data)

    return dirty_data

def remove_neurons(datum, angles,sfs, animal, count = False):
    neurons_to_keep = []
    data = resort_preprocessing(datum,angles,sfs,animal)
    number_neurons = data.shape[0]
    for i in range(number_neurons):
        stim_average = np.mean(data[i, :, :, :, 40:80], axis = 3) 
        best_sf = np.argmax(np.nanmean(stim_average, axis = (0,2))).astype('int')
        best_angle = np.argmax(np.nanmean(stim_average[:,best_sf,:], axis = 1)).astype('int')
        averaged_calcium = np.nanmean(stim_average[best_angle,best_sf,:])

        grey_data = data[i, :, :, :, 0:20]
        grey_average = np.mean(grey_data, axis = 3)
        best_sf = np.argmax(np.nanmean(grey_average, axis = (0,2))).astype(int)
        best_angle = np.argmax(np.nanmean(grey_average[:,best_sf,:], axis = 1)).astype(int)
        average_grey = np.nanmean(grey_average[best_angle,best_sf,:])
        std_grey = np.nanstd(grey_average[best_angle,best_sf,:])

        if np.abs(averaged_calcium - average_grey) >= 1.69*std_grey:
            neurons_to_keep.append(i)

    data_filtered = data[neurons_to_keep, :, :,:,:]
    if count:
        return data_filtered.shape[0]/data.shape[0]

    return data_filtered


## Figure 1


### Figure 1a: Example activity traces


In [ ]:
angle_tuned_to = 7               # One-based angle condition
response_window = slice(40, 80)    # Two-second response window
plot_angle_indices = np.arange(4, 9)  # Six conditions: indices 3–8
min_selectivity = 0.3

def find_tuned_neurons(
    data,
    angle_number,
    response_window,
    min_selectivity=0.15,
):
    """
    Return neurons whose preferred response is the requested angle
    and whose selectivity exceeds min_selectivity.

    data shape:
        (neurons, angles, SFs, trials, time)
    """
    target_index = angle_number - 1

    if data.ndim != 5:
        raise ValueError(
            "Data must have shape "
            "(neurons, angles, SFs, trials, time)."
        )

    if not 0 <= target_index < data.shape[1]:
        raise ValueError(
            f"angle_number must be between 1 and {data.shape[1]}."
        )

    angle_responses = np.nanmean(
        data[..., response_window],
        axis=(2, 3, 4),
    )

    valid_neurons = np.any(np.isfinite(angle_responses), axis=1)

    safe_responses = np.where(
        np.isfinite(angle_responses),
        angle_responses,
        -np.inf,
    )
    preferred_angles = np.argmax(safe_responses, axis=1)

    target_response = angle_responses[:, target_index]

    other_angles = np.arange(data.shape[1]) != target_index
    mean_other_response = np.nanmean(
        angle_responses[:, other_angles],
        axis=1,
    )

    denominator = (
        np.abs(target_response)
        + np.abs(mean_other_response)
        + np.finfo(float).eps
    )
    selectivity = (
        target_response - mean_other_response
    ) / denominator

    tuned_mask = (
        valid_neurons
        & np.isfinite(target_response)
        & np.isfinite(selectivity)
        & (preferred_angles == target_index)
        & (target_response > 0)
        & (selectivity >= min_selectivity)
    )

    tuned_neurons = np.flatnonzero(tuned_mask)

    return tuned_neurons, selectivity, angle_responses

def find_best_matched_pair(
    fr_neurons,
    ctr_neurons,
    fr_angle_responses,
    ctr_angle_responses,
    angle_indices,
):
    """
    Find the FR–CTR neuron pair with the smallest RMSE between
    their raw mean response profiles across the selected angles.
    """
    fr_profiles = fr_angle_responses[
        fr_neurons[:, None],
        angle_indices,
    ]
    ctr_profiles = ctr_angle_responses[
        ctr_neurons[:, None],
        angle_indices,
    ]

    valid_fr = np.all(np.isfinite(fr_profiles), axis=1)
    valid_ctr = np.all(np.isfinite(ctr_profiles), axis=1)

    fr_neurons = fr_neurons[valid_fr]
    ctr_neurons = ctr_neurons[valid_ctr]
    fr_profiles = fr_profiles[valid_fr]
    ctr_profiles = ctr_profiles[valid_ctr]

    if len(fr_neurons) == 0:
        raise ValueError("No valid tuned FR neurons were found.")

    if len(ctr_neurons) == 0:
        raise ValueError("No valid tuned CTR neurons were found.")

    best_rmse = np.inf
    best_fr_position = None
    best_ctr_position = None

    for fr_position, fr_profile in enumerate(fr_profiles):
        rmse = np.sqrt(
            np.mean(
                (ctr_profiles - fr_profile[None, :]) ** 2,
                axis=1,
            )
        )

        ctr_position = np.argmin(rmse)

        if rmse[ctr_position] < best_rmse:
            best_rmse = rmse[ctr_position]
            best_fr_position = fr_position
            best_ctr_position = ctr_position

    return (
        fr_neurons[best_fr_position],
        ctr_neurons[best_ctr_position],
        fr_profiles[best_fr_position],
        ctr_profiles[best_ctr_position],
        best_rmse,
    )

fr_animal = FOOD_RESTRICTED_HUNGRY[1]
fr_data = remove_neurons(
    HUNGRY_DECONV,
    HUNGRY_ANGLE,
    HUNGRY_SF,
    fr_animal,
)

ctr_animal = CONTROL_HUNGRY[4]
ctr_data = remove_neurons(
    HUNGRY_DECONV,
    HUNGRY_ANGLE,
    HUNGRY_SF,
    ctr_animal,
)

fr_tuned_neurons, fr_selectivity, fr_angle_responses = (
    find_tuned_neurons(
        fr_data,
        angle_number=angle_tuned_to,
        response_window=response_window,
        min_selectivity=min_selectivity,
    )
)

ctr_tuned_neurons, ctr_selectivity, ctr_angle_responses = (
    find_tuned_neurons(
        ctr_data,
        angle_number=angle_tuned_to,
        response_window=response_window,
        min_selectivity=min_selectivity,
    )
)

(
    fr_neuron,
    ctr_neuron,
    fr_mean_by_angle,
    ctr_mean_by_angle,
    matching_rmse,
) = find_best_matched_pair(
    fr_tuned_neurons,
    ctr_tuned_neurons,
    fr_angle_responses,
    ctr_angle_responses,
    plot_angle_indices,
)

fr_plot_data = np.nanmean(
    fr_data[
        fr_neuron,
        plot_angle_indices,
        :,
        :,
        response_window,
    ],
    axis=(1, 2),
)

ctr_plot_data = np.nanmean(
    ctr_data[
        ctr_neuron,
        plot_angle_indices,
        :,
        :,
        response_window,
    ],
    axis=(1, 2),
)


In [ ]:
frame_rate = 20  # Hz: 40 samples correspond to 2 seconds
time = np.arange(fr_plot_data.shape[1]) / frame_rate

fig, axes = plt.subplots(
    1,
    len(plot_angle_indices),
    figsize=(2.2, 2.2),
    sharex=True,
    sharey=True,
    gridspec_kw={"wspace": 0.5},
)

for panel, angle_index in enumerate(plot_angle_indices):
    ax = axes[panel]

    ax.plot(
        time,
        fr_plot_data[panel],
        color=COLORS["FR"],
        linewidth=1.4,
        label="FR",
        solid_capstyle="round",
    )
    ax.plot(
        time,
        ctr_plot_data[panel],
        color=COLORS["CTR"],
        linewidth=1.4,
        label="CTR",
        solid_capstyle="round",
    )

    ax.set_xlim(time[0], time[-1])
    ax.tick_params(
        axis="both",
        labelsize=7,
        width=0.7,
        length=2.5,
        direction="out",
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if panel > 0:
        ax.spines["left"].set_visible(False)
        ax.tick_params(
            axis="y",
            left=False,
            labelleft=False,
        )

plt.show()


### Figure 1b: PCA projection and trial covariance


In [ ]:
from sklearn.decomposition import PCA
from matplotlib.patches import Ellipse

plt.style.use(STYLE_PATH)

def lighten_color(color, amount=0.65):
    """
    Mix a color with white.
    amount=0 gives original color, amount=1 gives white.
    """
    import matplotlib.colors as mc

    c = np.array(mc.to_rgb(color))
    white = np.array([1, 1, 1])
    return tuple((1 - amount) * c + amount * white)

def ellipse_stats(points, level=0.95):
    points = points[np.all(np.isfinite(points), axis=1)]

    if points.shape[0] < 2:
        return None

    mean = np.mean(points, axis=0)
    cov = np.cov(points, rowvar=False)

    chi2_val = 5.991464547  # 95% chi-square, 2 d.o.f.

    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.maximum(eigvals, 0)

    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
    width, height = 2 * np.sqrt(eigvals * chi2_val)

    return mean, eigvals, eigvecs, angle, width, height

def PCA_matlab_like(data):
    Tdim=data.shape[0]
    Ndim=data.shape[1]
    pca = PCA(n_components=Ndim)
    if Ndim > Tdim:
        pca = PCA(n_components=Tdim)
    pca.fit(data)
    coeff = pca.components_.T
    score = pca.transform(data)
    eigs = pca.explained_variance_
    if Ndim > Tdim:
        u,s,v = np.linalg.svd(coeff)
        y = u[:,Tdim:]
        coeff = np.hstack([coeff, y])
        eigs = np.append(eigs, np.zeros(Ndim - Tdim))
        score = np.hstack([score, np.zeros((Tdim, Ndim - Tdim))])

    return coeff, score, eigs

def add_covariance_ellipse(
    ax,
    points,
    color,
    alpha=0.16,
    linewidth=0.8,
    zorder=2,
):
    stats = ellipse_stats(points)

    if stats is None:
        return None

    mean, eigvals, eigvecs, angle, width, height = stats

    ell = Ellipse(
        xy=mean,
        width=width,
        height=height,
        angle=angle,
        facecolor=color,
        edgecolor=color,
        alpha=alpha,
        linewidth=linewidth,
        zorder=zorder,
    )
    ax.add_patch(ell)

    return stats

def projection_manifold_with_trial_covariance(data, angles, sfs, animal, start, stop):
    data = resort_preprocessing(data, angles, sfs, animal)
    data = data[:, :, :, :, start:stop]

    number_neurons, n_angles, n_sfs, n_trials, n_time = data.shape

    data_averaged = np.nanmean(data, axis=(2, 3))  # neurons x angles x time

    mean_temporal_profile = np.nanmean(data_averaged, axis=1, keepdims=True)
    data_centered = data_averaged - mean_temporal_profile

    variance_over_time = np.nanvar(data_centered, axis=2, keepdims=True)
    scale = np.sqrt(variance_over_time)
    scale[scale == 0] = np.nan

    data_norm = data_centered / scale

    X_mean = data_norm.reshape(number_neurons, -1).T  # angle*time x neurons

    coeff, score_mean, eigs = PCA_matlab_like(X_mean)

    pca_mean = np.mean(X_mean, axis=0, keepdims=True)

    trial_data_centered = (
        data - mean_temporal_profile[:, :, None, None, :]
    ) / scale[:, :, None, None, :]

    trial_data_centered = np.nanmean(trial_data_centered, axis=2)

    trial_points = []
    trial_angle_index = []

    for angle_idx in range(n_angles):
        for trial_idx in range(n_trials):
            trial_traj = trial_data_centered[:, angle_idx, trial_idx, :].T

            if np.all(np.isnan(trial_traj)):
                continue

            valid_time = np.all(np.isfinite(trial_traj), axis=1)
            if np.sum(valid_time) == 0:
                continue

            trial_score = (trial_traj[valid_time] - pca_mean) @ coeff

            trial_point = np.mean(trial_score[:, :2], axis=0)

            trial_points.append(trial_point)
            trial_angle_index.append(angle_idx)

    trial_points = np.array(trial_points)
    trial_angle_index = np.array(trial_angle_index)

    return coeff, score_mean, eigs, trial_points, trial_angle_index
animal = 5
start = 50
stop = 80

coeff, score_mean, eigs, trial_points, trial_angle_index = (
    projection_manifold_with_trial_covariance(
        SATED_DECONV,
        SATED_ANGLE,
        SATED_SF,
        animal,
        start,
        stop,
    )
)

stimuli_index = np.repeat(np.arange(12), stop - start)

fig, ax = plt.subplots(figsize=(mm_to_inch(45), mm_to_inch(40)))

highlight_angles = [0, 1]

highlight_colors = {
    0: COLORS["Theory"],
    1: COLORS["Simulation"],
}

rest_colors = {
        2: COLORS["Other"],
        3: COLORS["compare"],
        4: COLORS["FR"],
        5: COLORS["CTR"],
}

for angle_idx in range(6):
    pts = trial_points[trial_angle_index == angle_idx]

    mean_traj = score_mean[stimuli_index == angle_idx, :2]
    mean_point = np.nanmean(mean_traj, axis=0)

    if angle_idx in highlight_angles:
        color = highlight_colors[angle_idx]
        point_alpha = 0.70
        ellipse_alpha = 0.18
        ellipse_lw = 0.9
        mean_size = 24
        trial_size = 8
        edgecolor = "black"
        edgewidth = 0.35
        z = 5
    else:
        base_color = rest_colors[angle_idx]
        color = lighten_color(base_color, amount=0.70)
        point_alpha = 0.45
        ellipse_alpha = 0.14
        ellipse_lw = 0.6
        mean_size = 14
        trial_size = 7
        edgecolor = "none"
        edgewidth = 0
        z = 2

    ax.scatter(
        pts[:, 0],
        pts[:, 1],
        s=trial_size,
        color=color,
        alpha=point_alpha,
        linewidth=0,
        zorder=z,
    )

    add_covariance_ellipse(
        ax,
        pts,
        color=color,
        alpha=ellipse_alpha,
        linewidth=ellipse_lw,
        zorder=z + 1,
    )

    ax.scatter(
        mean_point[0],
        mean_point[1],
        s=mean_size,
        color=color,
        edgecolor=edgecolor,
        linewidth=edgewidth,
        zorder=z + 3,
    )

ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_aspect("equal", adjustable="datalim")

clean_axis(ax)
ax.locator_params(axis="x", nbins=4)
ax.locator_params(axis="y", nbins=4)

fig.tight_layout(pad=0.3)

plt.show()


### Figure 1c: Noise correlation

One value per animal, using SATED grey-screen settings and an animal-label
permutation test.


In [ ]:
from itertools import combinations

def noise_corr_at_time(data, timebin, min_trials=5):
    """Raw correlation matrix and mean off-diagonal Fisher z at one time bin."""
    response = data[..., timebin]  # neurons x angles x SFs x trials
    residual_blocks = []

    for iangle in range(response.shape[1]):
        for isf in range(response.shape[2]):
            X = response[:, iangle, isf, :]
            X = X[:, np.all(np.isfinite(X), axis=0)]
            if X.shape[1] >= min_trials:
                residual_blocks.append(X - X.mean(axis=1, keepdims=True))

    if not residual_blocks:
        raise ValueError(f"No stimulus has {min_trials} complete trials")

    residuals = np.concatenate(residual_blocks, axis=1)
    if residuals.shape[0] < 2:
        raise ValueError("At least two selected neurons are required")
    if np.any(np.var(residuals, axis=1, ddof=1) <= 0):
        raise ValueError(f"A neuron has zero residual variance at time bin {timebin}")

    corr = np.corrcoef(residuals)
    pair_r = corr[np.triu_indices_from(corr, k=1)]
    if not np.all(np.isfinite(pair_r)):
        raise ValueError(f"Non-finite correlation at time bin {timebin}")
    mean_z = np.mean(np.arctanh(np.clip(pair_r, -0.999999, 0.999999)))
    return corr, mean_z

def animal_noise_corr_timecourse(datum, angles, sfs, animal, min_trials=5):
    """Mean Fisher-z noise correlation at every time bin for one animal."""
    data = remove_neurons(datum, angles, sfs, animal)
    mean_z = [
        noise_corr_at_time(data, timebin, min_trials)[1]
        for timebin in range(data.shape[-1])
    ]
    return np.asarray(mean_z)

def exact_animal_permutation(fr_values, ctr_values):
    """Exact two-sided test of the FR-minus-control difference in animal means."""
    values = np.concatenate([fr_values, ctr_values])
    observed = fr_values.mean() - ctr_values.mean()
    null = []

    for fr_idx in combinations(range(len(values)), len(fr_values)):
        is_fr = np.zeros(len(values), dtype=bool)
        is_fr[list(fr_idx)] = True
        null.append(values[is_fr].mean() - values[~is_fr].mean())

    null = np.asarray(null)
    tolerance = 10 * np.finfo(float).eps
    p_value = np.mean(np.abs(null) >= np.abs(observed) - tolerance)
    return observed, p_value


In [ ]:
NC_STATE = "SATED"
BOX_WINDOW = "grey"  # "post" (80:120) or "grey" (0:25 plus 90:120)
NC_MIN_TRIALS = 5

nc_datum, nc_angles, nc_sfs = SATED_DECONV, SATED_ANGLE, SATED_SF
nc_fr_ids, nc_ctr_ids = FOOD_RESTRICTED_SATED, CONTROL_SATED

fr_z_time = np.vstack([
    animal_noise_corr_timecourse(
        nc_datum, nc_angles, nc_sfs, animal, NC_MIN_TRIALS
    )
    for animal in nc_fr_ids
])
ctr_z_time = np.vstack([
    animal_noise_corr_timecourse(
        nc_datum, nc_angles, nc_sfs, animal, NC_MIN_TRIALS
    )
    for animal in nc_ctr_ids
])

if NC_STATE == "SATED":
    assert fr_z_time.shape == (8, 120)
    assert ctr_z_time.shape == (6, 120)
assert np.all(np.isfinite(fr_z_time))
assert np.all(np.isfinite(ctr_z_time))

box_windows = {
    "post": np.arange(80, 120),
    "grey": np.r_[0:40, 80:120],
    "bar": np.arange(40, 80)
}
window_labels = {
    "post": "Post-stimulus (bins 80-120)",
    "bar": "Bar presentation (bins 40-80)",
    "grey": "Grey screen (bins 0-40 + 80-120)",
}
if BOX_WINDOW not in box_windows:
    raise ValueError('BOX_WINDOW must be "post" or "grey"')

box_bins = box_windows[BOX_WINDOW]
fr_box_z = fr_z_time[:, box_bins].mean(axis=1)
ctr_box_z = ctr_z_time[:, box_bins].mean(axis=1)
box_difference, box_p = exact_animal_permutation(fr_box_z, ctr_box_z)

print(f"{NC_STATE}, {window_labels[BOX_WINDOW]}")
print(
    f"Back-transformed mean r: FR={np.tanh(fr_box_z.mean()):.4f}, "
    f"control={np.tanh(ctr_box_z.mean()):.4f}"
)
print(f"FR - control (Fisher z)={box_difference:.4f}")
print(f"Exact two-sided animal-label permutation p={box_p:.4g}")

plot_rng = np.random.default_rng(9)
groups = [np.tanh(ctr_box_z), np.tanh(fr_box_z)]
colors = [COLORS["CTR"], COLORS["FR"]]

fig, ax = plt.subplots(figsize=(mm_to_inch(30), mm_to_inch(35)))

boxes = ax.boxplot(
    groups,
    positions=[0, 1],
    widths=0.48,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "black", "lw": 0.8},
    whiskerprops={"color": "black", "lw": 0.7},
    capprops={"color": "black", "lw": 0.7},
    boxprops={"color": "black", "lw": 0.7},
)

for patch, color in zip(boxes["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)

for x, values, color in zip([0, 1], groups, colors):
    jitter = plot_rng.uniform(-0.10, 0.10, len(values))
    ax.scatter(
        x + jitter,
        values,
        s=14,
        color=color,
        edgecolor="black",
        linewidth=0.3,
        zorder=3,
    )

y_min = min(groups[0].min(), groups[1].min())
y_max = max(groups[0].max(), groups[1].max())
y_range = max(y_max - y_min, 1e-3)

bracket_y = y_max + 0.10 * y_range
bracket_h = 0.04 * y_range

ax.plot(
    [0, 0, 1, 1],
    [bracket_y, bracket_y + bracket_h,
     bracket_y + bracket_h, bracket_y],
    color="black",
    lw=0.7,
    clip_on=False,
)

p_text = r"$p<0.001$" if box_p < 0.001 else rf"$p={box_p:.3f}$"
ax.text(
    0.5,
    bracket_y + bracket_h + 0.02 * y_range,
    p_text,
    ha="center",
    va="bottom",
)

ax.set_xticks([0, 1])
ax.set_xticklabels(["CTR", "FR"])
ax.set_ylabel("Noise correlation")
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(y_min - 0.10 * y_range, y_max + 0.30 * y_range)

clean_axis(ax)
fig.tight_layout(pad=0.3)

plt.show()


### Figure 1d: Dimensionality (participation ratio)


In [ ]:
TIME_WINDOW = (40,80) 
def _safe_nanmean(arr, axis):
    count = np.sum(np.isfinite(arr), axis=axis)
    total = np.nansum(arr, axis=axis)
    return np.divide(
        total,
        count,
        out=np.full(np.shape(total), np.nan, dtype=float),
        where=count > 0,
    )

def participation_ratio_from_evals(evals, eps=1e-12):
    evals = np.asarray(evals, dtype=float)
    evals = evals[np.isfinite(evals)]
    evals = np.clip(evals, 0, None)
    evals[evals < eps] = 0

    total_var = np.sum(evals)
    if total_var <= 0:
        return np.nan

    return total_var**2 / np.sum(evals**2)

def participation_ratio_per_animal(data, time_window=TIME_WINDOW, ddof=1):
    if time_window is not None:
        t0, t1 = time_window
        data = data[..., t0:t1]

    _, n_angles, n_sfs, _, n_times = data.shape

    pr = np.full((n_angles, n_sfs), np.nan)

    for iangle in range(n_angles):
        for jsf in range(n_sfs):

            dat = data[:, iangle, jsf, :, :]

            valid_neurons = np.any(np.isfinite(dat), axis=(1, 2))
            dat = dat[valid_neurons]

            if dat.shape[0] == 0:
                continue

            pr_over_time = []

            for itime in range(n_times):
                dat_t = dat[:, :, itime]

                valid_trials = np.all(np.isfinite(dat_t), axis=0)
                dat_t = dat_t[:, valid_trials].T  # trials x neurons

                if dat_t.shape[0] <= ddof or dat_t.shape[1] == 0:
                    continue

                try:
                    cov_t = np.atleast_2d(np.cov(dat_t, rowvar=False, ddof=ddof))
                    evals_t = np.linalg.eigvalsh(cov_t)
                except np.linalg.LinAlgError:
                    continue

                pr_t = participation_ratio_from_evals(evals_t)
                pr_over_time.append(pr_t)

            if len(pr_over_time) == 0:
                continue

            pr[iangle, jsf] = _safe_nanmean(np.asarray(pr_over_time), axis=0)

    return _safe_nanmean(pr, axis=(0, 1))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fr_pr = np.array([
    participation_ratio_per_animal(
        resort_preprocessing(SATED_DECONV, SATED_ANGLE, SATED_SF, animal)
    )
    for animal in FOOD_RESTRICTED_SATED
])

ctr_pr = np.array([
    participation_ratio_per_animal(
        resort_preprocessing(SATED_DECONV, SATED_ANGLE, SATED_SF, animal)
    )
    for animal in CONTROL_SATED
])

def permutation_pvalue_independent(x, y, n_perm=10000, seed=0):
    rng = np.random.default_rng(seed)

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x) & (x > 0)]
    y = y[np.isfinite(y) & (y > 0)]

    if len(x) < 2 or len(y) < 2:
        return np.nan

    x = np.log(x)
    y = np.log(y)

    observed = np.abs(np.mean(x) - np.mean(y))

    pooled = np.concatenate([x, y])
    n_x = len(x)

    null = np.empty(n_perm)

    for i in range(n_perm):
        shuffled = rng.permutation(pooled)
        x_perm = shuffled[:n_x]
        y_perm = shuffled[n_x:]
        null[i] = np.abs(np.mean(x_perm) - np.mean(y_perm))

    p = (np.sum(null >= observed) + 1) / (n_perm + 1)
    return p

p_pr = permutation_pvalue_independent(ctr_pr, fr_pr, n_perm=10000, seed=0)
print(f"Permutation p-value: {p_pr:.4g}")

plt.style.use(STYLE_PATH)

plot_data = [
    ctr_pr[np.isfinite(ctr_pr)],
    fr_pr[np.isfinite(fr_pr)],
]
positions = [0, 1]
colors = [COLORS["CTR"], COLORS["FR"]]
plot_rng = np.random.default_rng(9)

fig, ax = plt.subplots(figsize=(mm_to_inch(30), mm_to_inch(35)))
boxes = ax.boxplot(
    plot_data,
    positions=positions,
    widths=0.48,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "black", "lw": 0.8},
    whiskerprops={"color": "black", "lw": 0.7},
    capprops={"color": "black", "lw": 0.7},
    boxprops={"color": "black", "lw": 0.7},
)
for patch, color in zip(boxes["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)

for x, values, color in zip(positions, plot_data, colors):
    jitter = plot_rng.uniform(-0.10, 0.10, len(values))
    ax.scatter(
        x + jitter,
        values,
        s=14,
        color=color,
        edgecolor="black",
        linewidth=0.3,
        zorder=3,
    )

y_min = min(plot_data[0].min(), plot_data[1].min())
y_max = max(plot_data[0].max(), plot_data[1].max())
y_range = max(y_max - y_min, 1e-3)
bracket_y = y_max + 0.10 * y_range
bracket_h = 0.04 * y_range
ax.plot(
    [0, 0, 1, 1],
    [bracket_y, bracket_y + bracket_h, bracket_y + bracket_h, bracket_y],
    color="black",
    lw=0.7,
    clip_on=False,
)
if np.isfinite(p_pr):
    p_text = r"$p<0.001$" if p_pr < 0.001 else rf"$p={p_pr:.3f}$"
else:
    p_text = "n.s."
ax.text(
    0.5,
    bracket_y + bracket_h + 0.02 * y_range,
    p_text,
    ha="center",
    va="bottom",
)

ax.set_xticks(positions)
ax.set_xticklabels(["CTR", "FR"])
ax.set_ylabel("Dimensionality")
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(y_min - 0.10 * y_range, y_max + 0.30 * y_range)
clean_axis(ax)
fig.tight_layout(pad=0.3)

plt.show()


### Figure 1e–f: Noise orientation and signal overlap (SNR)

The original plotting function draws **1e: Noise orientation** on the left and
**1f: Signal overlap (labelled SNR in the paper)** on the right.


In [ ]:
def compute_signal_vectors(response, t0=40, t1=80):
    """
    Compute Δμ for each ROI, SF, and pair of angles.
    response: (n_rois, n_angles, n_sfs, n_trials, n_time)
    Returns:
      Δμ: (n_rois, n_angles, n_angles, n_sfs)
    """
    resp = np.nanmean(response[..., t0:t1], axis=-1)  # → (n_rois, n_angles, n_sfs, n_trials)
    mean_resp = np.nanmean(resp, axis=-1)             # → (n_rois, n_angles, n_sfs)
    dm = mean_resp[:, :, np.newaxis, :] - mean_resp[:, np.newaxis, :, :]

    return dm

def compute_preferred_sf(response):
    return np.argmax(np.nanmean(response[:,:,:,:,40:80], axis=(1,3,4)), axis=1) # Shape Nrois

def compute_noise_metrics_all(response, signal_vectors, total_k, t0=40, t1=80):
    """
    response:       (n_rois, n_angles, n_sfs, n_trials, n_time)
    signal_vectors: (n_rois, n_angles, n_angles, n_sfs)

    Returns dict with:
      dot_prod       (n_angles, n_angles-1, n_sfs, total_k)
      top_evals      (n_angles, n_sfs, total_k)
      noise_corr     (n_angles, n_sfs)
      mean_dist      (n_angles, n_angles-1, n_sfs)
      pc_coeff_pref  (n_rois, n_angles, n_sfs, total_k)
    """
    resp = np.nanmean(response[..., t0:t1], axis=-1)  # → (n_rois, n_angles, n_sfs, n_trials)
    n_rois, n_angles, _, n_trials = resp.shape

    if total_k is None:
        kmax = MIN_NEURONS
        repeats = REPEATS
    else:
        repeats = REPEATS
        kmax = total_k
    n_sfs = N_SFS
    dot_prod   = np.full((n_angles, n_angles-1, n_sfs, kmax, repeats), np.nan)
    norm =       np.full((n_angles, n_angles-1, n_sfs, repeats), np.nan)
    snr_pc     = np.full((n_angles, n_angles-1, n_sfs, repeats), np.nan)
    top_evals  = np.full((n_angles, n_sfs, kmax, repeats), np.nan)
    sfs = compute_preferred_sf(response)

    for r in range(repeats):
        if total_k is None:
            rng = np.random.default_rng(RNG_SEED+r)
            idx = rng.choice(n_rois, kmax, replace=False)
            resp_small = resp[idx]
            sig_vec = signal_vectors[idx]
            sfs_small = sfs[idx]
        else:
            resp_small = resp
            sig_vec = signal_vectors
            sfs_small = sfs

        for sf in range(n_sfs):
            for ia in range(n_angles):
                if N_SFS ==1:
                    dat = np.zeros((resp_small.shape[0], resp_small.shape[3]))
                    for i in range(resp_small.shape[0]):
                        best_sf = sfs_small[i]  
                        dat[i, :] = resp_small[i, ia, best_sf, :]
                else:
                    dat = resp_small[:, ia, sf, :]  # (n_rois, n_trials)

                dat_T = dat.T                         # (n_trials, n_rois)
                covariance_matrix     = np.cov(dat_T, rowvar=False, ddof=0)  # (n_rois, n_rois)
                try:
                    evals, evecs = np.linalg.eigh(covariance_matrix)
                except np.linalg.LinAlgError:
                    continue
                order = np.argsort(evals)[::-1]
                sorted_evals = evals[order]
                sorted_evecs = evecs[:, order]
                top_evals[ia, sf, :, r] = sorted_evals[:kmax]

                others = (ia + np.arange(1, n_angles)) % n_angles

                for j, ja in enumerate(others):
                    if N_SFS == 1:
                        sv = sig_vec[np.arange(resp_small.shape[0]), ia, ja, sfs_small]         # (Nsub,)
                    else:
                        sv = sig_vec[:, ia, ja, sf]                       # (Nsub,)
                    snr_pc[ia, j, sf, r] = 0# Dont do directly anymore
                    norm[ia, j, sf, r] = np.linalg.norm(sv)

                    for k in range(kmax):
                        pc = sorted_evecs[:, k]
                        dot_prod[ia, j, sf, k, r] = np.power((np.dot(sv, pc)),2)
    return {
        'dot_prod'     : dot_prod,
        'top_evals'    : top_evals,
        'snr_pc'       : snr_pc,
        'norm'         : norm

    }

def analyze_animal(subject_id, deconv, angles, sfs, subject_key, total_k, start= 40, stop=80):
    """
    Wrap everything for one animal.
    subject_key is the identifier you pass into your resort_preprocessing.
    """
    data = resort_preprocessing(deconv, angles, sfs, subject_key)

    sig_vec = compute_signal_vectors(data, t0 = start, t1 = stop)

    metrics = compute_noise_metrics_all(data, sig_vec, total_k, t0 = start, t1 = stop)

    metrics['subject'] = subject_id
    metrics['n_rois']   = data.shape[0]

    return metrics

RNG_SEED = 1
REPEATS = 1
TOTAL_K = 13
N_SFS = 5

t0 = 40
t1 = 80

number_neurons = []
for i in range(14):
    x =resort_preprocessing(SATED_DECONV, SATED_ANGLE, SATED_SF, i)
    number_neurons.append(x.shape[0])

MIN_NEURONS = min(number_neurons)
angles = SATED_ANGLE       # e.g. array of 12 angles
sfs    = SATED_SF          # e.g. number or list of SFs
fr_metrics  = []
for i, subj in enumerate(FOOD_RESTRICTED_SATED):
    fr_metrics.append(
        analyze_animal(subject_id=f"FR_{i}",
                       deconv=SATED_DECONV,
                       angles=angles,
                       sfs=sfs,
                       subject_key=subj,
                       total_k=TOTAL_K, start=t0, stop=t1)
    )

ctr_metrics = []
for j, subj in enumerate(CONTROL_SATED):
    ctr_metrics.append(
        analyze_animal(subject_id=f"CTR_{j}",
                       deconv=SATED_DECONV,
                       angles=angles,
                       sfs=sfs,
                       subject_key=subj,
                       total_k=TOTAL_K, start=t0, stop=t1)
    )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use(STYLE_PATH)

def nansem(x, axis=0):
    """SEM ignoring NaNs, with correct per-point sample count."""
    x = np.asarray(x, dtype=float)
    n = np.sum(np.isfinite(x), axis=axis)
    sem = np.nanstd(x, axis=axis, ddof=1) / np.sqrt(n)
    return np.where(n > 1, sem, np.nan)

def _valid_and_sort(x, y, sem, sort_by_x=True):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    sem = np.asarray(sem, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(sem)

    x = x[valid]
    y = y[valid]
    sem = sem[valid]

    if sort_by_x:
        order = np.argsort(x)
        x = x[order]
        y = y[order]
        sem = sem[order]

    return x, y, sem

def extract_eigen_cos2_overlap(metrics_list, angle_neighbor=0):
    """
    Extract per-animal eigenvalues, cos²(theta), raw overlap, squared signal
    norm, and total SNR.

    metrics_list: list of animal metric dicts
    angle_neighbor:
        0 means use the first neighboring angle in your ordering:
        others = (ia + np.arange(1, n_angles)) % n_angles

    Returns
    -------
    evals_plot   : (n_animals, total_k)
    cos2_plot    : (n_animals, total_k)
    overlap_plot : (n_animals, total_k)
    norm2_plot   : (n_animals,)
    total_snr    : (n_animals,)
    """

    total_k = metrics_list[0]["top_evals"].shape[-2]
    n_anim = len(metrics_list)

    evals_plot = np.full((n_anim, total_k), np.nan)
    cos2_plot = np.full((n_anim, total_k), np.nan)
    overlap_plot = np.full((n_anim, total_k), np.nan)
    norm2_plot = np.full(n_anim, np.nan)
    total_snr = np.full(n_anim, np.nan)

    eps = 1e-12

    for i, m in enumerate(metrics_list):
        if m is None:
            continue

        top_evals = m["top_evals"]

        evals_plot[i] = np.nanmean(top_evals, axis=(0, 1, 3))

        overlap = m["dot_prod"]

        signal_norm = m["norm"]

        overlap_neighbor = overlap[:, angle_neighbor, :, :, :]

        norm_neighbor = signal_norm[:, angle_neighbor, :, :]
        norm2_neighbor = norm_neighbor ** 2
        norm2_plot[i] = np.nanmean(norm2_neighbor)

        overlap_plot[i] = np.nanmean(overlap_neighbor, axis=(0, 1, 3))

        norm2_by_mode = norm2_neighbor[:, :, None, :]
        cos2_neighbor = np.divide(
            overlap_neighbor,
            norm2_by_mode,
            out=np.full_like(overlap_neighbor, np.nan, dtype=float),
            where=norm2_by_mode > eps,
        )

        cos2_plot[i] = np.nanmean(cos2_neighbor, axis=(0, 1, 3))

        valid_evals = np.where(top_evals > eps, top_evals, np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            snr_by_mode = (
                cos2_neighbor * norm2_by_mode / valid_evals
            )
        valid_observation = np.any(np.isfinite(snr_by_mode), axis=2)
        snr_neighbor = np.nansum(snr_by_mode, axis=2)
        snr_neighbor[~valid_observation] = np.nan
        total_snr[i] = np.nanmean(snr_neighbor)

    return evals_plot, cos2_plot, overlap_plot, norm2_plot, total_snr

def plot_eigen_cos2_and_overlap_stacked(
    x_fr,
    cos2_fr,
    cos2_sem_fr,
    overlap_fr,
    overlap_sem_fr,
    x_ctr,
    cos2_ctr,
    cos2_sem_ctr,
    overlap_ctr,
    overlap_sem_ctr,
    xlabel=r"$\left\langle \lambda_i \right\rangle$",
    cos2_ylabel=r"$\left\langle \cos^2 \theta_i \right\rangle$",
    overlap_ylabel=r"$\left\langle (\nu_i \cdot \Delta \mu)^2 \right\rangle$",
    save_path="final_figures/panels/eigenvalue_cos2_overlap_FR_CTR.svg",
    xscale="linear",
    yscale_cos2="linear",
    yscale_overlap="linear",
    sort_by_x=True,
    layout="stacked",
):
    """
    Two-panel plot with either a stacked or side-by-side layout.

    Top panel: cos²(theta)
    Bottom panel: raw overlap
    Shared x-axis: averaged eigenvalue
    """

    if layout == "side_by_side":
        fig, axes = plt.subplots(
            1,
            2,
            figsize=(mm_to_inch(100), mm_to_inch(70)),
            sharex=True,
            gridspec_kw={"width_ratios": [1, 1], "wspace": 0.42},
        )
    elif layout == "stacked":
        fig, axes = plt.subplots(
            2,
            1,
            figsize=(mm_to_inch(100), mm_to_inch(70)),
            sharex=True,
            gridspec_kw={"height_ratios": [1, 1], "hspace": 0.12},
        )
    else:
        raise ValueError("layout must be 'stacked' or 'side_by_side'.")

    ax_cos2, ax_overlap = axes

    x_ctr_cos2, cos2_ctr, cos2_sem_ctr = _valid_and_sort(
        x_ctr, cos2_ctr, cos2_sem_ctr, sort_by_x=sort_by_x
    )
    x_fr_cos2, cos2_fr, cos2_sem_fr = _valid_and_sort(
        x_fr, cos2_fr, cos2_sem_fr, sort_by_x=sort_by_x
    )

    x_ctr_overlap, overlap_ctr, overlap_sem_ctr = _valid_and_sort(
        x_ctr, overlap_ctr, overlap_sem_ctr, sort_by_x=sort_by_x
    )
    x_fr_overlap, overlap_fr, overlap_sem_fr = _valid_and_sort(
        x_fr, overlap_fr, overlap_sem_fr, sort_by_x=sort_by_x
    )

    ax_cos2.fill_between(
        x_ctr_cos2,
        cos2_ctr - cos2_sem_ctr,
        cos2_ctr + cos2_sem_ctr,
        color=COLORS["CTR"],
        alpha=0.16,
        linewidth=0,
        zorder=1,
    )

    ax_cos2.plot(
        x_ctr_cos2,
        cos2_ctr,
        "o-",
        color=COLORS["CTR"],
        linewidth=1.0,
        markersize=3,
        markerfacecolor=COLORS["CTR"],
        markeredgecolor="black",
        markeredgewidth=0.3,
        label="CTR",
        zorder=3,
    )

    ax_cos2.fill_between(
        x_fr_cos2,
        cos2_fr - cos2_sem_fr,
        cos2_fr + cos2_sem_fr,
        color=COLORS["FR"],
        alpha=0.16,
        linewidth=0,
        zorder=2,
    )

    ax_cos2.plot(
        x_fr_cos2,
        cos2_fr,
        "o-",
        color=COLORS["FR"],
        linewidth=1.0,
        markersize=3,
        markerfacecolor=COLORS["FR"],
        markeredgecolor="black",
        markeredgewidth=0.3,
        label="FR",
        zorder=4,
    )

    ax_cos2.set_ylabel(cos2_ylabel)
    ax_cos2.set_xscale(xscale)
    ax_cos2.set_yscale(yscale_cos2)
    ax_cos2.locator_params(axis="y", nbins=4)

    clean_axis(ax_cos2)

    ax_cos2.legend(
        frameon=False,
        loc="best",
        handlelength=1.5,
        handletextpad=0.4,
        borderaxespad=0.2,
    )

    if layout == "side_by_side":
        ax_cos2.set_xlabel(xlabel)
        ax_cos2.locator_params(axis="x", nbins=4)
    else:
        ax_cos2.tick_params(labelbottom=False)

    ax_overlap.fill_between(
        x_ctr_overlap,
        overlap_ctr - overlap_sem_ctr,
        overlap_ctr + overlap_sem_ctr,
        color=COLORS["CTR"],
        alpha=0.16,
        linewidth=0,
        zorder=1,
    )

    ax_overlap.plot(
        x_ctr_overlap,
        overlap_ctr,
        "o-",
        color=COLORS["CTR"],
        linewidth=1.0,
        markersize=3,
        markerfacecolor=COLORS["CTR"],
        markeredgecolor="black",
        markeredgewidth=0.3,
        label="CTR",
        zorder=3,
    )

    ax_overlap.fill_between(
        x_fr_overlap,
        overlap_fr - overlap_sem_fr,
        overlap_fr + overlap_sem_fr,
        color=COLORS["FR"],
        alpha=0.16,
        linewidth=0,
        zorder=2,
    )

    ax_overlap.plot(
        x_fr_overlap,
        overlap_fr,
        "o-",
        color=COLORS["FR"],
        linewidth=1.0,
        markersize=3,
        markerfacecolor=COLORS["FR"],
        markeredgecolor="black",
        markeredgewidth=0.3,
        label="FR",
        zorder=4,
    )

    ax_overlap.set_xlabel(xlabel)
    ax_overlap.set_ylabel(overlap_ylabel)

    ax_overlap.set_xscale(xscale)
    ax_overlap.set_yscale(yscale_overlap)

    ax_overlap.locator_params(axis="x", nbins=4)
    ax_overlap.locator_params(axis="y", nbins=4)

    clean_axis(ax_overlap)

    fig.tight_layout(pad=0.3)

    if save_path is not None:
        fig.savefig(
            save_path,
            transparent=True,
            bbox_inches="tight",
            pad_inches=0.02,
        )

    plt.show()

    return fig, axes

fr_evals_plot, fr_cos2_plot, fr_overlap_plot, fr_norm2_plot, fr_total_snr = (
    extract_eigen_cos2_overlap(
    fr_metrics,
    angle_neighbor=0,
    )
)

ctr_evals_plot, ctr_cos2_plot, ctr_overlap_plot, ctr_norm2_plot, ctr_total_snr = (
    extract_eigen_cos2_overlap(
    ctr_metrics,
    angle_neighbor=0,
    )
)

x_fr = np.nanmean(fr_evals_plot, axis=0)
x_ctr = np.nanmean(ctr_evals_plot, axis=0)

cos2_fr = np.nanmean(fr_cos2_plot, axis=0)
cos2_ctr = np.nanmean(ctr_cos2_plot, axis=0)

overlap_fr = np.nanmean(fr_overlap_plot, axis=0)
overlap_ctr = np.nanmean(ctr_overlap_plot, axis=0)

cos2_sem_fr = nansem(fr_cos2_plot, axis=0)
cos2_sem_ctr = nansem(ctr_cos2_plot, axis=0)

overlap_sem_fr = nansem(fr_overlap_plot, axis=0)
overlap_sem_ctr = nansem(ctr_overlap_plot, axis=0)

fig, axes = plot_eigen_cos2_and_overlap_stacked(
    x_fr=x_fr,
    cos2_fr=cos2_fr,
    cos2_sem_fr=cos2_sem_fr,
    overlap_fr=overlap_fr,
    overlap_sem_fr=overlap_sem_fr,
    x_ctr=x_ctr,
    cos2_ctr=cos2_ctr,
    cos2_sem_ctr=cos2_sem_ctr,
    overlap_ctr=overlap_ctr,
    overlap_sem_ctr=overlap_sem_ctr,
    xlabel=r"$\left\langle \lambda_k \right\rangle$",
    cos2_ylabel=r"$\left\langle \cos^2 \phi_k \right\rangle$",
    overlap_ylabel=r"$\left\langle (\boldsymbol{\nu}_k \cdot \boldsymbol{\Delta \mu})^2 \right\rangle$",
    save_path=None,
    layout="side_by_side",
)


## Extended Data Figure 1

Run after Figure 1 to reuse the data and helper functions.


### ED Figure 1a: Noise correlation over time — food restriction and recovery


In [ ]:
RECOVERY_DECONV = np.load((RAW_DATA_DIR / 'predictions_fullTrace_recovery.npy'), allow_pickle=True)

AngStim_data = (RAW_DATA_DIR / 'metadata_deconv/stimAngle_recovery.mat')
ANG_STIM_DATA = loadmat(AngStim_data, simplify_cells= True)
RECOVERY_ANGLE = ANG_STIM_DATA['order_of_stim_arossAnimals']

SfStim_data = (RAW_DATA_DIR / 'metadata_deconv/stimSpatFreq_recovery.mat')
SF_STIM_DATA = loadmat(SfStim_data, simplify_cells= True)
RECOVERY_SF = SF_STIM_DATA['stimSpatFreq_arossAnimals']

FOOD_RESTRICTED_RECOVERY = [0,2,3,6,8,9,11,12]
CONTROL_RECOVERY         = [1,4,5,7,10,13]


In [ ]:
for NC_STATE in ["SATED", "RECOVERY"]:
    NC_MIN_TRIALS = 5
    NC_FRAME_RATE = 20.0
    NC_STIM_ONSET = 40
    NC_BAR_STOP = 80

    if NC_STATE == "HUNGRY":
        nc_datum, nc_angles, nc_sfs = HUNGRY_DECONV, HUNGRY_ANGLE, HUNGRY_SF
        nc_fr_ids, nc_ctr_ids = FOOD_RESTRICTED_HUNGRY, CONTROL_HUNGRY
    elif NC_STATE == "SATED":
        nc_datum, nc_angles, nc_sfs = SATED_DECONV, SATED_ANGLE, SATED_SF
        nc_fr_ids, nc_ctr_ids = FOOD_RESTRICTED_SATED, CONTROL_SATED
    elif NC_STATE == "RECOVERY":
        nc_datum, nc_angles, nc_sfs = RECOVERY_DECONV, RECOVERY_ANGLE, RECOVERY_SF
        nc_fr_ids, nc_ctr_ids = FOOD_RESTRICTED_RECOVERY, CONTROL_RECOVERY
    else:
        raise ValueError(f"Unknown NC_STATE: {NC_STATE}")

    fr_z_time = np.vstack([
        animal_noise_corr_timecourse(
            nc_datum, nc_angles, nc_sfs, animal, NC_MIN_TRIALS
        )
        for animal in nc_fr_ids
    ])
    ctr_z_time = np.vstack([
        animal_noise_corr_timecourse(
            nc_datum, nc_angles, nc_sfs, animal, NC_MIN_TRIALS
        )
        for animal in nc_ctr_ids
    ])

    if NC_STATE == "SATED":
        assert fr_z_time.shape == (8, 120)
        assert ctr_z_time.shape == (6, 120)
    assert np.all(np.isfinite(fr_z_time))
    assert np.all(np.isfinite(ctr_z_time))

    time_order = np.r_[
        NC_STIM_ONSET:fr_z_time.shape[1],
        0:NC_STIM_ONSET,
    ]
    time_s = np.arange(fr_z_time.shape[1]) / NC_FRAME_RATE

    plt.style.use(STYLE_PATH)
    fig, ax = plt.subplots(figsize=(mm_to_inch(65), mm_to_inch(38)))

    for scores, color, label in [
        (ctr_z_time, COLORS["CTR"], "CTR"),
        (fr_z_time, COLORS["FR"], "FR"),
    ]:
        scores = scores[:, time_order]

        mean_z = scores.mean(axis=0)
        sem_z = scores.std(axis=0, ddof=1) / np.sqrt(len(scores))

        ax.plot(
            time_s,
            np.tanh(mean_z),
            color=color,
            label=label,
            lw=1.0,
        )
        ax.fill_between(
            time_s,
            np.tanh(mean_z - sem_z),
            np.tanh(mean_z + sem_z),
            color=color,
            alpha=0.20,
            linewidth=0,
        )

    ax.axvspan(
        (NC_BAR_STOP - NC_STIM_ONSET) / NC_FRAME_RATE,
        6,
        color="0.85",
        alpha=0.50,
        zorder=0,
    )
    ax.axvline(0, color="0.5", lw=0.6, ls="--")
    ax.set_ylabel("Noise correlation")
    ax.set_xlabel("Time from stimulus onset (s)")
    ax.set_xticks([0, 2, 4, 6])
    ax.set_xlim(0, 6)

    ax.legend(
        frameon=False,
        loc="upper right",
        handlelength=1.3,
        handletextpad=0.4,
        borderaxespad=0.2,
    )

    clean_axis(ax)
    fig.tight_layout(pad=0.3)

    plt.show()


### ED Figure 1b: Covariance eigenvalue spectrum


In [ ]:
N_MODES = 15
EIGEN_EPS = 1e-8
TIME_WINDOW = (40,80)  # set to (40, 80) for evoked-response covariance only

def _safe_nansem(arr, axis=0):
    arr = np.asarray(arr, dtype=float)
    count = np.sum(np.isfinite(arr), axis=axis)
    mean = _safe_nanmean(arr, axis=axis)
    centered = arr - np.expand_dims(mean, axis)
    ss = np.nansum(centered**2, axis=axis)
    var = np.divide(
        ss,
        count - 1,
        out=np.full(np.shape(ss), np.nan, dtype=float),
        where=count > 1,
    )
    sd = np.sqrt(var)
    return np.divide(
        sd,
        np.sqrt(count),
        out=np.full(np.shape(sd), np.nan, dtype=float),
        where=count > 0,
    )

def eigenvalues_per_mode(
    data,
    n_modes=N_MODES,
    min_eval=EIGEN_EPS,
    time_window=TIME_WINDOW,
    ddof=1,
):
    if time_window is not None:
        t0, t1 = time_window
        data = data[..., t0:t1]

    n_neurons, n_angles, n_sfs, n_trials, n_times = data.shape
    output = np.full((n_modes, n_angles, n_sfs), np.nan)

    for iangle in range(n_angles):
        for jsf in range(n_sfs):

            dat = data[:, iangle, jsf, :, :]

            valid_neurons = np.any(np.isfinite(dat), axis=(1, 2))
            dat = dat[valid_neurons]

            if dat.shape[0] == 0:
                continue

            covs = []

            for itime in range(n_times):
                dat_t = dat[:, :, itime]

                valid_trials = np.all(np.isfinite(dat_t), axis=0)
                dat_t = dat_t[:, valid_trials].T  # trials x neurons

                if dat_t.shape[0] <= ddof:
                    continue

                cov_t = np.atleast_2d(np.cov(dat_t, rowvar=False, ddof=ddof))
                covs.append(cov_t)

            if len(covs) == 0:
                continue

            evals_over_time = []

            for cov_t in covs:
                try:
                    eigenvals_t = np.linalg.eigvalsh(cov_t)
                except np.linalg.LinAlgError:
                    continue

                eigenvals_t = np.sort(eigenvals_t)[::-1]
                eigenvals_t[eigenvals_t < min_eval] = np.nan

                evals_t = np.full(n_modes, np.nan)
                n_store = min(n_modes, eigenvals_t.size)
                evals_t[:n_store] = eigenvals_t[:n_store]

                evals_over_time.append(evals_t)

            if len(evals_over_time) == 0:
                continue

            eigenvals = _safe_nanmean(np.vstack(evals_over_time), axis=0)

            output[:, iangle, jsf] = eigenvals

    return _safe_nanmean(output, axis=(1, 2))

fr_evals = np.vstack([
    eigenvalues_per_mode(remove_neurons(SATED_DECONV, SATED_ANGLE, SATED_SF, FR))
    for FR in FOOD_RESTRICTED_SATED
])
ctr_evals = np.vstack([
    eigenvalues_per_mode(remove_neurons(SATED_DECONV, SATED_ANGLE, SATED_SF, CTR))
    for CTR in CONTROL_SATED
])

fr_mean = _safe_nanmean(fr_evals, axis=0)
fr_sem = _safe_nansem(fr_evals, axis=0)
ctr_mean = _safe_nanmean(ctr_evals, axis=0)
ctr_sem = _safe_nansem(ctr_evals, axis=0)
eval_ratio = np.divide(
    fr_mean,
    ctr_mean,
    out=np.full_like(fr_mean, np.nan, dtype=float),
    where=np.isfinite(fr_mean) & np.isfinite(ctr_mean) & (ctr_mean > 0),
)
eval_ratio_sem = np.full_like(eval_ratio, np.nan, dtype=float)

valid = (
    np.isfinite(eval_ratio)
    & np.isfinite(fr_mean)
    & np.isfinite(ctr_mean)
    & np.isfinite(fr_sem)
    & np.isfinite(ctr_sem)
    & (fr_mean > 0)
    & (ctr_mean > 0)
)

eval_ratio_sem[valid] = eval_ratio[valid] * np.sqrt(
    (fr_sem[valid] / fr_mean[valid]) ** 2
    + (ctr_sem[valid] / ctr_mean[valid]) ** 2
)
mode_limit = 10
fr_mean = fr_mean[:mode_limit]
fr_sem = fr_sem[:mode_limit]
ctr_mean = ctr_mean[:mode_limit]
ctr_sem = ctr_sem[:mode_limit]
eval_ratio = eval_ratio[:mode_limit]
modes = np.arange(1, mode_limit+1)

valid_ratio = np.isfinite(eval_ratio)

plt.style.use(STYLE_PATH)

def permutation_pvalue_independent(x, y, n_perm=10000, seed=0):
    """
    Two-sided independent-sample permutation test.
    Tests difference in mean log eigenvalue between groups.
    """
    rng = np.random.default_rng(seed)

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid_x = np.isfinite(x) & (x > 0)
    valid_y = np.isfinite(y) & (y > 0)

    x = np.log(x[valid_x])
    y = np.log(y[valid_y])

    if len(x) < 2 or len(y) < 2:
        return np.nan

    observed = np.abs(np.mean(x) - np.mean(y))

    pooled = np.concatenate([x, y])
    n_x = len(x)

    perm_diffs = np.empty(n_perm)

    for i in range(n_perm):
        shuffled = rng.permutation(pooled)
        x_perm = shuffled[:n_x]
        y_perm = shuffled[n_x:]
        perm_diffs[i] = np.abs(np.mean(x_perm) - np.mean(y_perm))

    p = (np.sum(perm_diffs >= observed) + 1) / (n_perm + 1)
    return p

pvals = np.full(mode_limit, np.nan)

for imode in range(mode_limit):
    pvals[imode] = permutation_pvalue_independent(
        fr_evals[:, imode],
        ctr_evals[:, imode],
        n_perm=10000,
        seed=imode,
    )

def p_to_stars(p):
    if not np.isfinite(p):
        return ""
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""

fig, ax = plt.subplots(figsize=(mm_to_inch(35), mm_to_inch(35)))

x = modes[valid_ratio]

ax.errorbar(
    x,
    fr_mean[valid_ratio],
    yerr=fr_sem[valid_ratio],
    fmt="o",
    color=COLORS["FR"],
    ecolor=COLORS["FR"],
    elinewidth=0.6,
    capsize=2,
    capthick=0.6,
    markersize=3,
    markerfacecolor=COLORS["FR"],
    markeredgecolor="black",
    markeredgewidth=0.3,
    linewidth=1.0,
    label="FR",
)

ax.errorbar(
    x,
    ctr_mean[valid_ratio],
    yerr=ctr_sem[valid_ratio],
    fmt="o",
    color=COLORS["CTR"],
    ecolor=COLORS["CTR"],
    elinewidth=0.6,
    capsize=2,
    capthick=0.6,
    markersize=3,
    markerfacecolor=COLORS["CTR"],
    markeredgecolor="black",
    markeredgewidth=0.3,
    linewidth=1.0,
    label="CTR",
)

star_ys = []

for imode, mode in enumerate(modes):
    if not valid_ratio[imode]:
        continue

    stars = p_to_stars(pvals[imode])
    if stars == "":
        continue

    y_top = np.nanmax([
        fr_mean[imode] + fr_sem[imode],
        ctr_mean[imode] + ctr_sem[imode],
    ])

    if not np.isfinite(y_top) or y_top <= 0:
        continue

    y_star = y_top
    star_ys.append(y_star)

    ax.text(
        mode,
        y_star,
        stars,
        ha="center",
        va="bottom",
        fontsize=7,
        color="black",
    )

ax.set_xlabel("Covariance eigenvalue mode")
ax.set_ylabel("Eigenvalue")
ax.set_yscale("log")

ax.locator_params(axis="x", nbins=5)
ax.tick_params(axis="both", which="major", length=2.5, width=0.6)
ax.tick_params(axis="both", which="minor", length=1.5, width=0.5)

clean_axis(ax)

ax.legend(
    frameon=False,
    handlelength=1.5,
    handletextpad=0.4,
    borderaxespad=0.2,
)

fig.tight_layout(pad=0.3)

plt.show()


### ED Figure 1c: Normal and trial-shuffled decoding accuracy

Tasks run sequentially.  shuffled controls use
`CONTROL_HUNGRY`.


In [ ]:
def removing_nc(data):
    rng = np.random.default_rng()

    for l in range(data.shape[0]):
        data[l,:,:,:,:] = rng.permuted(data[l,:,:,:,:], axis=2)
    return data
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def trimming(datum):
    return_data = np.full((datum.shape[0],datum.shape[1],datum.shape[2],datum.shape[3]),np.nan)
    for neuron in range(datum.shape[0]):
        for angle in range(datum.shape[1]):
            for sf in range(datum.shape[2]):
                trial_responses = datum[neuron, angle, sf,:,40:80]
                mean_response = np.nanmean(trial_responses)
                std_response = np.nanstd(trial_responses)
                for trial in range(datum.shape[3]):
                    value = np.nanmean(datum[neuron,angle,sf,trial,40:80],axis = -1)
                    if np.abs(value - mean_response) >  2* std_response:
                        return_data[neuron,angle,sf,trial] = np.nan
                    else:
                        return_data[neuron,angle,sf,trial] = value
    return return_data
def LDA_RESPONSE_minimal_sf(data):
    """
    Leave-one-trial-out shrinkage-LDA decoder.

    Parameters
    ----------
    data : array
        Shape: (neurons, directions, spatial_frequencies, trials)

    Returns
    -------
    float
        Mean orientation-decoding accuracy across spatial frequencies.
        With 12 directions wrapped to 6 orientations, chance = 1/6.
    """
    data = np.asarray(data, dtype=float)

    if data.ndim != 4:
        raise ValueError(
            "data must have shape "
            "(neurons, directions, spatial_frequencies, trials)"
        )

    nrois, ndirections, nspatfreqs, ntrials = data.shape

    if ndirections % 2 != 0:
        raise ValueError("The number of directions must be even.")

    norientations = ndirections // 2
    accuracy_per_sf = []

    for sf in range(nspatfreqs):
        n_correct = 0
        n_total = 0

        for test_trial in range(ntrials):

            train = np.delete(
                data[:, :, sf, :],
                test_trial,
                axis=-1,
            )

            X_train = train.transpose(1, 2, 0).reshape(
                -1, nrois
            )

            y_train = np.repeat(
                np.arange(ndirections),
                train.shape[-1],
            )

            X_test = data[:, :, sf, test_trial].T

            train_nonempty = np.any(
                np.isfinite(X_train),
                axis=1,
            )
            X_train = X_train[train_nonempty]
            y_train = y_train[train_nonempty]

            feature_nonempty = np.any(
                np.isfinite(X_train),
                axis=0,
            )
            X_train = X_train[:, feature_nonempty]
            X_test = X_test[:, feature_nonempty]

            if X_train.shape[1] == 0:
                continue

            test_nonempty = np.any(
                np.isfinite(X_test),
                axis=1,
            )

            if not np.any(test_nonempty):
                continue

            class_counts = np.bincount(
                y_train,
                minlength=ndirections,
            )

            if np.any(class_counts < 2):
                continue

            model = make_pipeline(
                SimpleImputer(strategy="median"),

                StandardScaler(),

                LinearDiscriminantAnalysis(
                    solver="lsqr",
                    shrinkage="auto",
                    priors=np.full(
                        ndirections,
                        1 / ndirections,
                    ),
                ),
            )

            model.fit(X_train, y_train)

            direction_probability = model.predict_proba(
                X_test[test_nonempty]
            )

            lda = model.named_steps[
                "lineardiscriminantanalysis"
            ]

            probability_full = np.zeros(
                (
                    direction_probability.shape[0],
                    ndirections,
                )
            )

            probability_full[:, lda.classes_.astype(int)] = (
                direction_probability
            )

            orientation_probability = (
                probability_full[:, :norientations]
                + probability_full[:, norientations:]
            )

            predicted_orientation = np.argmax(
                orientation_probability,
                axis=1,
            )

            actual_direction = np.arange(ndirections)[
                test_nonempty
            ]
            actual_orientation = (
                actual_direction % norientations
            )

            n_correct += np.sum(
                predicted_orientation == actual_orientation
            )
            n_total += len(actual_orientation)

        if n_total > 0:
            accuracy_per_sf.append(
                n_correct / n_total
            )

    if not accuracy_per_sf:
        return np.nan

    return float(np.mean(accuracy_per_sf))
def process_data(args):
    index, group, shuffle = args

    if shuffle:
        datum_deconv = removing_nc(remove_neurons(SATED_DECONV, SATED_ANGLE, SATED_SF, index))
    else:
        datum_deconv = remove_neurons(SATED_DECONV, SATED_ANGLE, SATED_SF, index)

    data = trimming(datum_deconv)
    accuracy = LDA_RESPONSE_minimal_sf(data)

    return group, accuracy, shuffle


In [ ]:
fd_args_normal = [(FD, 'FOOD_RESTRICTED', False) for FD in FOOD_RESTRICTED_SATED]
cn_args_normal = [(CN, 'CONTROL', False) for CN in CONTROL_SATED]
fd_args_shuffle = [(FD, 'FOOD_RESTRICTED', True) for FD in FOOD_RESTRICTED_SATED]
cn_args_shuffle = [(CN, 'CONTROL', True) for CN in CONTROL_HUNGRY]
all_args = fd_args_normal + cn_args_normal + fd_args_shuffle + cn_args_shuffle

results = [process_data(args) for args in all_args]

fd_decoding_normal_accuracy = []
ctr_decoding_normal_accuracy = []
fd_decoding_shuffle_accuracy = []
ctr_decoding_shuffle_accuracy = []

for group, accuracy, shuffle in results:
    if group == 'FOOD_RESTRICTED' and not shuffle:
        fd_decoding_normal_accuracy.append(accuracy)
    elif group == 'CONTROL' and not shuffle:
        ctr_decoding_normal_accuracy.append(accuracy)
    elif group == 'FOOD_RESTRICTED' and shuffle:
        fd_decoding_shuffle_accuracy.append(accuracy)
    elif group == 'CONTROL' and shuffle:
        ctr_decoding_shuffle_accuracy.append(accuracy)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

plt.style.use(STYLE_PATH)

rng = np.random.default_rng(42)

def clean_values(x):
    x = np.asarray(x, dtype=float)
    return x[np.isfinite(x)]

def significance_label(p):
    if p >= 0.05:
        return "NS"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    return "*"

def add_significance_bracket(ax, x1, x2, y, height, label):
    ax.plot(
        [x1, x1, x2, x2],
        [y, y + height, y + height, y],
        color="black",
        lw=0.7,
        clip_on=False,
    )
    ax.text(
        (x1 + x2) / 2,
        y + height,
        label,
        ha="center",
        va="bottom",
        fontsize=6,
    )

def draw_boxplots(ax, data, positions, colors, widths=0.55):
    bp = ax.boxplot(
        data,
        positions=positions,
        widths=widths,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "black", "linewidth": 0.8},
        whiskerprops={"color": "black", "linewidth": 0.7},
        capprops={"color": "black", "linewidth": 0.7},
        boxprops={"edgecolor": "black", "linewidth": 0.7},
    )

    for box, color in zip(bp["boxes"], colors):
        box.set_facecolor(color)
        box.set_alpha(0.5)

    for position, values, color in zip(positions, data, colors):
        jitter = rng.uniform(-0.10, 0.10, size=len(values))
        ax.scatter(
            position + jitter,
            values,
            s=10,
            facecolor=color,
            edgecolor="white",
            linewidth=0.35,
            zorder=3,
        )

fr_normal = clean_values(fd_decoding_normal_accuracy)
ctr_normal = clean_values(ctr_decoding_normal_accuracy)
fr_shuffle = clean_values(fd_decoding_shuffle_accuracy)
ctr_shuffle = clean_values(ctr_decoding_shuffle_accuracy)
fig, ax = plt.subplots(
    figsize=(mm_to_inch(70), mm_to_inch(42))
)

positions = [0, 0.75, 1.8, 2.55]

draw_boxplots(
    ax,
    data=[
        fr_normal,
        ctr_normal,
        fr_shuffle,
        ctr_shuffle,
    ],
    positions=positions,
    colors=[
        COLORS["FR"],
        COLORS["CTR"],
        COLORS["FR"],
        COLORS["CTR"],
    ],
    widths=0.50,
)

ax.set_xticks(
    [(positions[0] + positions[1]) / 2,
     (positions[2] + positions[3]) / 2]
)
ax.set_xticklabels(["Normal", "Trial shuffled"])
ax.set_ylabel("Decoding accuracy")
ax.set_xlim(-0.45, 3.0)

for x, label, color in zip(
    positions,
    ["FR", "Control", "FR", "Control"],
    [COLORS["FR"], COLORS["CTR"]] * 2,
):
    ax.text(
        x,
        -0.16,
        label,
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=6,
        color=color,
    )

bracket_y = max(
    np.nanmax(fr_normal),
    np.nanmax(ctr_normal),
    np.nanmax(fr_shuffle),
    np.nanmax(ctr_shuffle),
) + 0.02
bracket_height = 0.02
p_normal = mannwhitneyu(fr_normal, ctr_normal, alternative="two-sided").pvalue
add_significance_bracket(
ax,
0,
0.8,
bracket_y,
bracket_height,
significance_label(p_normal),
)

fig.subplots_adjust(bottom=0.25)
plt.show()


## Extended Data Figure 2


### ED Figure 2a: Noise orientation and signal overlap across sessions — angles


In [ ]:
ALL_STATE_MODE_COUNT = 9
ALL_STATE_T0 = 40
ALL_STATE_T1 = 80
ALL_STATE_EIGEN_EPS = 1e-8
ALL_STATE_ANGLE_NEIGHBOR = 0  # first cyclic neighbor = 30 degrees

if N_SFS != 5 or REPEATS != 1:
    raise RuntimeError("Figure 1f expects N_SFS=5 and REPEATS=1 from Figure 1e.")

ALL_STATE_CONFIGS = {
    "SATED": {
        "deconv": SATED_DECONV,
        "angles": SATED_ANGLE,
        "sfs": SATED_SF,
        "FR": FOOD_RESTRICTED_SATED,
        "CTR": CONTROL_SATED,
    },
    "HUNGRY": {
        "deconv": HUNGRY_DECONV,
        "angles": HUNGRY_ANGLE,
        "sfs": HUNGRY_SF,
        "FR": FOOD_RESTRICTED_HUNGRY,
        "CTR": CONTROL_HUNGRY,
    },
    "RECOVERY": {
        "deconv": RECOVERY_DECONV,
        "angles": RECOVERY_ANGLE,
        "sfs": RECOVERY_SF,
        "FR": FOOD_RESTRICTED_RECOVERY,
        "CTR": CONTROL_RECOVERY,
    },
}

all_state_metrics = {}

for state_name, state_config in ALL_STATE_CONFIGS.items():
    metrics_for_state = {}

    for group_name in ("FR", "CTR"):
        animal_ids = state_config[group_name]
        expected_count = 8 if group_name == "FR" else 6
        if len(animal_ids) != expected_count:
            raise AssertionError(
                f"{state_name} {group_name}: expected {expected_count} animals, "
                f"found {len(animal_ids)}."
            )

        print(f"Computing {state_name} {group_name}: {len(animal_ids)} animals")
        metrics_for_group = []

        for animal_index, subject_key in enumerate(animal_ids):
            animal_metrics = analyze_animal(
                subject_id=f"{state_name}_{group_name}_{animal_index}",
                deconv=state_config["deconv"],
                angles=state_config["angles"],
                sfs=state_config["sfs"],
                subject_key=subject_key,
                total_k=ALL_STATE_MODE_COUNT,
                start=ALL_STATE_T0,
                stop=ALL_STATE_T1,
            )

            expected_metric_shapes = {
                "top_evals": (12, 5, ALL_STATE_MODE_COUNT, REPEATS),
                "dot_prod": (12, 11, 5, ALL_STATE_MODE_COUNT, REPEATS),
                "norm": (12, 11, 5, REPEATS),
            }
            for metric_name, expected_metric_shape in expected_metric_shapes.items():
                actual_metric_shape = animal_metrics[metric_name].shape
                if actual_metric_shape != expected_metric_shape:
                    raise AssertionError(
                        f"{state_name} {group_name} animal {subject_key} "
                        f"{metric_name}: expected shape {expected_metric_shape}, "
                        f"found {actual_metric_shape}."
                    )
            metrics_for_group.append(animal_metrics)

        if len(metrics_for_group) != expected_count or any(
            metric is None for metric in metrics_for_group
        ):
            raise AssertionError(f"Incomplete metrics for {state_name} {group_name}.")

        metrics_for_state[group_name] = metrics_for_group

    all_state_metrics[state_name] = metrics_for_state

all_state_summaries = {}

for state_name, metrics_for_state in all_state_metrics.items():
    summaries_for_state = {}

    for group_name in ("FR", "CTR"):
        metrics_for_group = metrics_for_state[group_name]
        evals_by_animal, cos2_by_animal, overlap_by_animal, _, _ = (
            extract_eigen_cos2_overlap(
                metrics_for_group,
                angle_neighbor=ALL_STATE_ANGLE_NEIGHBOR,
            )
        )

        expected_shape = (len(metrics_for_group), ALL_STATE_MODE_COUNT)
        for metric_name, metric_values in (
            ("eigenvalue", evals_by_animal),
            ("cos2", cos2_by_animal),
            ("overlap", overlap_by_animal),
        ):
            if metric_values.shape != expected_shape:
                raise AssertionError(
                    f"{state_name} {group_name} {metric_name}: expected shape "
                    f"{expected_shape}, found {metric_values.shape}."
                )
            if not np.any(np.isfinite(metric_values)):
                raise AssertionError(
                    f"{state_name} {group_name} {metric_name} has no finite values."
                )

        finite_cos2 = cos2_by_animal[np.isfinite(cos2_by_animal)]
        finite_overlap = overlap_by_animal[np.isfinite(overlap_by_animal)]

        if np.any(finite_cos2 < -1e-10) or np.any(finite_cos2 > 1.0 + 1e-8):
            raise AssertionError(f"{state_name} {group_name}: cos^2 is outside [0, 1].")
        if np.any(finite_overlap < -1e-12):
            raise AssertionError(f"{state_name} {group_name}: overlap is negative.")

        valid_eigen_by_animal = (
            np.isfinite(evals_by_animal)
            & (evals_by_animal > ALL_STATE_EIGEN_EPS)
        )
        evals_for_summary = np.where(valid_eigen_by_animal, evals_by_animal, np.nan)
        cos2_for_summary = np.where(valid_eigen_by_animal, cos2_by_animal, np.nan)
        overlap_for_summary = np.where(
            valid_eigen_by_animal,
            overlap_by_animal,
            np.nan,
        )

        summaries_for_state[group_name] = {
            "evals_by_animal_raw": evals_by_animal,
            "cos2_by_animal_raw": cos2_by_animal,
            "overlap_by_animal_raw": overlap_by_animal,
            "evals_by_animal": evals_for_summary,
            "cos2_by_animal": cos2_for_summary,
            "overlap_by_animal": overlap_for_summary,
            "x": np.nanmean(evals_for_summary, axis=0),
            "cos2_mean": np.nanmean(cos2_for_summary, axis=0),
            "cos2_sem": nansem(cos2_for_summary, axis=0),
            "overlap_mean": np.nanmean(overlap_for_summary, axis=0),
            "overlap_sem": nansem(overlap_for_summary, axis=0),
        }

    all_state_summaries[state_name] = summaries_for_state


In [ ]:
from matplotlib.lines import Line2D

ALL_STATE_STYLES = {
    "SATED": {"linestyle": "-", "marker": "o"},
    "HUNGRY": {"linestyle": "--", "marker": "*"},
    "RECOVERY": {"linestyle": ":", "marker": "^"},
}

def prepare_all_state_curve(summary, value_key, sem_key):
    """Filter numerical zero modes and sort a group/state curve by eigenvalue."""
    x = np.asarray(summary["x"], dtype=float)
    y = np.asarray(summary[value_key], dtype=float)
    sem = np.asarray(summary[sem_key], dtype=float)

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
        & np.isfinite(sem)
        & (x > ALL_STATE_EIGEN_EPS)
    )

    if not np.any(valid):
        raise ValueError(f"No valid modes remain for {value_key}.")

    x = x[valid]
    y = y[valid]
    sem = sem[valid]
    order = np.argsort(x)

    return x[order], y[order], sem[order]

def plot_eigen_alignment_all_states(
    summaries,
    save_path,
):
    """Plot all three behavioral states for FR and CTR on two shared-x axes."""
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(mm_to_inch(100), mm_to_inch(70)),
        sharex=True,
        gridspec_kw={"height_ratios": [1, 1], "hspace": 0.12},
    )
    ax_cos2, ax_overlap = axes

    panel_specs = (
        (ax_cos2, "cos2_mean", "cos2_sem"),
        (ax_overlap, "overlap_mean", "overlap_sem"),
    )

    for ax, value_key, sem_key in panel_specs:
        for state_index, (state_name, state_style) in enumerate(
            ALL_STATE_STYLES.items()
        ):
            for group_index, group_name in enumerate(("CTR", "FR")):
                x, y, sem = prepare_all_state_curve(
                    summaries[state_name][group_name],
                    value_key,
                    sem_key,
                )
                color = COLORS[group_name]
                zorder = 2 + state_index + group_index * 0.1

                ax.fill_between(
                    x,
                    y - sem,
                    y + sem,
                    color=color,
                    alpha=0.08,
                    linewidth=0,
                    zorder=zorder,
                )
                ax.plot(
                    x,
                    y,
                    color=color,
                    linestyle=state_style["linestyle"],
                    marker=state_style["marker"],
                    linewidth=1.0,
                    markersize=3,
                    markerfacecolor=color,
                    markeredgecolor="black",
                    markeredgewidth=0.3,
                    label=f"{state_name} {group_name}",
                    zorder=zorder + 1,
                )

        clean_axis(ax)
        ax.locator_params(axis="y", nbins=4)

    ax_cos2.set_ylabel(r"$\left\langle \cos^2 \phi_k \right\rangle$")
    ax_cos2.tick_params(labelbottom=False)

    ax_overlap.set_xlabel(r"$\left\langle \lambda_k \right\rangle$")
    ax_overlap.set_ylabel(
        r"$\left\langle (\boldsymbol{\nu}_k \cdot \boldsymbol{\Delta \mu})^2 \right\rangle$"
    )
    ax_overlap.locator_params(axis="x", nbins=4)

    fig.align_ylabels(axes)
    fig.subplots_adjust(
        left=0.17,
        right=0.98,
        bottom=0.14,
        top=0.98,
        hspace=0.16,
    )

    group_handles = [
        Line2D(
            [0],
            [0],
            color=COLORS[group_name],
            marker="o",
            linewidth=1.0,
            markersize=3,
            markeredgecolor="black",
            markeredgewidth=0.3,
            label=group_name,
        )
        for group_name in ("CTR", "FR")
    ]
    state_handles = [
        Line2D(
            [0],
            [0],
            color="0.2",
            linestyle=state_style["linestyle"],
            marker=state_style["marker"],
            markerfacecolor="white",
            markeredgecolor="black",
            markeredgewidth=0.3,
            linewidth=1.0,
            markersize=3,
            label=state_name,
        )
        for state_name, state_style in ALL_STATE_STYLES.items()
    ]

    group_legend = ax_cos2.legend(
        handles=group_handles,
        frameon=False,
        loc="upper left",
        handlelength=1.5,
        handletextpad=0.4,
        borderaxespad=0.2,
    )
    ax_cos2.add_artist(group_legend)
    ax_cos2.legend(
        handles=state_handles,
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        handlelength=1.5,
        handletextpad=0.4,
        borderaxespad=0.2,
    )

    plt.show()

    return fig, axes

all_state_fig, all_state_axes = plot_eigen_alignment_all_states(
    all_state_summaries,
    save_path=None,
)


### ED Figure 2b: Noise orientation and signal overlap — spatial frequencies


In [ ]:
SF_ALIGNMENT_MODE_COUNT = 9
SF_ALIGNMENT_T0 = 40
SF_ALIGNMENT_T1 = 80
SF_ALIGNMENT_EIGEN_EPS = 1e-8
SF_ALIGNMENT_NORM_EPS = 1e-12
SF_ALIGNMENT_DIRECTED_PAIRS = np.asarray(
    [
        (1, 2),
        (2, 1),
        (2, 3),
        (3, 2),
        (3, 4),
        (4, 3),
        (4, 5),
        (5, 4),
    ],
    dtype=int,
)

if SF_ALIGNMENT_DIRECTED_PAIRS.shape != (8, 2):
    raise AssertionError("Figure 1g requires exactly eight directed SF pairs.")
if np.unique(SF_ALIGNMENT_DIRECTED_PAIRS, axis=0).shape[0] != 8:
    raise AssertionError("Figure 1g directed SF pairs must be unique.")
if np.any(np.abs(np.diff(SF_ALIGNMENT_DIRECTED_PAIRS, axis=1)) != 1):
    raise AssertionError("Every Figure 1g SF pair must be adjacent.")
if any(set(pair) == {1, 5} for pair in SF_ALIGNMENT_DIRECTED_PAIRS):
    raise AssertionError("Figure 1g must not wrap SF 5 back to SF 1.")

def sf_nanmean(values, axis=0):
    """NaN-aware mean that stays quiet when one animal has no valid mode."""
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    count = np.sum(finite, axis=axis)
    total = np.sum(np.where(finite, values, 0.0), axis=axis)
    return np.divide(
        total,
        count,
        out=np.full(np.shape(total), np.nan, dtype=float),
        where=count > 0,
    )

def sf_nansem(values, axis=0):
    """Animal SEM with an independent finite count for every mode."""
    values = np.asarray(values, dtype=float)
    n_finite = np.sum(np.isfinite(values), axis=axis)
    with np.errstate(invalid="ignore", divide="ignore"):
        sem = np.nanstd(values, axis=axis, ddof=1) / np.sqrt(n_finite)
    return np.where(n_finite > 1, sem, np.nan)

def sf_compute_animal_metrics(subject_id, subject_key):
    """Compute nine source-SF covariance modes for every adjacent directed pair."""
    data = resort_preprocessing(
        SATED_DECONV,
        SATED_ANGLE,
        SATED_SF,
        subject_key,
    )
    if data.ndim != 5:
        raise AssertionError(f"{subject_id}: expected a five-dimensional response array.")

    n_rois, n_angles, n_sfs, _, n_time = data.shape
    if (n_angles, n_sfs) != (12, 5):
        raise AssertionError(
            f"{subject_id}: expected 12 angles and 5 SFs, got {(n_angles, n_sfs)}."
        )
    if n_rois < SF_ALIGNMENT_MODE_COUNT:
        raise AssertionError(
            f"{subject_id}: {n_rois} ROIs cannot supply 9 covariance modes."
        )
    if n_time < SF_ALIGNMENT_T1:
        raise AssertionError(
            f"{subject_id}: only {n_time} bins are available for bins 40-80."
        )

    sf_condition_means = np.full((n_rois, n_angles, n_sfs), np.nan)
    sf_source_eigenvalues = np.full(
        (n_sfs, n_angles, SF_ALIGNMENT_MODE_COUNT),
        np.nan,
    )
    sf_source_eigenvectors = np.full(
        (n_sfs, n_angles, n_rois, SF_ALIGNMENT_MODE_COUNT),
        np.nan,
    )
    sf_valid_trials = np.zeros((n_sfs, n_angles), dtype=int)

    for sf_index in range(n_sfs):
        for angle_index in range(n_angles):
            trial_window = data[
                :,
                angle_index,
                sf_index,
                :,
                SF_ALIGNMENT_T0:SF_ALIGNMENT_T1,
            ]
            complete_trials = np.all(np.isfinite(trial_window), axis=(0, 2))
            trial_responses = np.mean(
                trial_window[:, complete_trials, :],
                axis=-1,
            )
            sf_valid_trials[sf_index, angle_index] = trial_responses.shape[1]
            if trial_responses.shape[1] < 2:
                continue

            sf_condition_means[:, angle_index, sf_index] = np.mean(
                trial_responses,
                axis=1,
            )
            covariance = np.cov(
                trial_responses.T,
                rowvar=False,
                ddof=0,
            )
            eigenvalues, eigenvectors = np.linalg.eigh(covariance)
            descending = np.argsort(eigenvalues)[::-1][:SF_ALIGNMENT_MODE_COUNT]
            sf_source_eigenvalues[sf_index, angle_index] = eigenvalues[descending]
            sf_source_eigenvectors[sf_index, angle_index] = eigenvectors[:, descending]

    if np.any(sf_valid_trials < 2):
        bad_sf, bad_angle = np.argwhere(sf_valid_trials < 2)[0]
        raise AssertionError(
            f"{subject_id}: SF {bad_sf + 1}, angle {bad_angle} has fewer than two "
            "complete trials."
        )

    pair_shape = (
        len(SF_ALIGNMENT_DIRECTED_PAIRS),
        n_angles,
        SF_ALIGNMENT_MODE_COUNT,
    )
    sf_pair_eigenvalues = np.full(pair_shape, np.nan)
    sf_overlap = np.full(pair_shape, np.nan)
    sf_cos2 = np.full(pair_shape, np.nan)
    sf_signal_norm = np.full(pair_shape[:2], np.nan)

    for pair_index, (source_sf, target_sf) in enumerate(
        SF_ALIGNMENT_DIRECTED_PAIRS
    ):
        source_index = source_sf - 1
        target_index = target_sf - 1
        for angle_index in range(n_angles):
            eigenvalues = sf_source_eigenvalues[source_index, angle_index]
            eigenvectors = sf_source_eigenvectors[source_index, angle_index]
            valid_eigenmodes = (
                np.isfinite(eigenvalues)
                & (eigenvalues > SF_ALIGNMENT_EIGEN_EPS)
                & np.all(np.isfinite(eigenvectors), axis=0)
            )
            signal = (
                sf_condition_means[:, angle_index, source_index]
                - sf_condition_means[:, angle_index, target_index]
            )
            if not np.all(np.isfinite(signal)):
                continue

            signal_norm = np.linalg.norm(signal)
            if not np.isfinite(signal_norm) or signal_norm <= SF_ALIGNMENT_NORM_EPS:
                continue
            sf_signal_norm[pair_index, angle_index] = signal_norm

            raw_overlap = np.square(eigenvectors.T @ signal)
            valid_alignment = valid_eigenmodes & np.isfinite(raw_overlap)
            sf_pair_eigenvalues[
                pair_index,
                angle_index,
                valid_alignment,
            ] = eigenvalues[valid_alignment]
            sf_overlap[
                pair_index,
                angle_index,
                valid_alignment,
            ] = raw_overlap[valid_alignment]
            sf_cos2[
                pair_index,
                angle_index,
                valid_alignment,
            ] = raw_overlap[valid_alignment] / signal_norm**2

    expected_pair_shape = (8, 12, SF_ALIGNMENT_MODE_COUNT)
    if sf_pair_eigenvalues.shape != expected_pair_shape:
        raise AssertionError(f"{subject_id}: invalid pair metric shape.")
    if sf_overlap.shape != expected_pair_shape or sf_cos2.shape != expected_pair_shape:
        raise AssertionError(f"{subject_id}: invalid alignment metric shape.")
    if sf_source_eigenvalues.shape != (5, 12, SF_ALIGNMENT_MODE_COUNT):
        raise AssertionError(f"{subject_id}: invalid source eigenvalue shape.")

    return {
        "subject": subject_id,
        "subject_key": subject_key,
        "n_rois": n_rois,
        "directed_pairs": SF_ALIGNMENT_DIRECTED_PAIRS.copy(),
        "condition_means": sf_condition_means,
        "source_eigenvalues": sf_source_eigenvalues,
        "source_eigenvectors": sf_source_eigenvectors,
        "valid_trials": sf_valid_trials,
        "pair_eigenvalues": sf_pair_eigenvalues,
        "signal_norm": sf_signal_norm,
        "overlap": sf_overlap,
        "cos2": sf_cos2,
    }

def sf_analyze_group(group_name, animal_ids):
    """Run the SATED SF analysis without reusing Figure 1e/1f metric objects."""
    group_metrics = []
    for animal_index, subject_key in enumerate(animal_ids):
        subject_id = f"{group_name}_{animal_index}"
        group_metrics.append(sf_compute_animal_metrics(subject_id, subject_key))
    return group_metrics

def sf_summarize_group(group_metrics):
    """Average pairs/angles within animal, then mean and SEM across animals."""
    sf_eigenvalues_by_animal = np.asarray(
        [
            sf_nanmean(metric["pair_eigenvalues"], axis=(0, 1))
            for metric in group_metrics
        ]
    )
    sf_cos2_by_animal = np.asarray(
        [sf_nanmean(metric["cos2"], axis=(0, 1)) for metric in group_metrics]
    )
    sf_overlap_by_animal = np.asarray(
        [sf_nanmean(metric["overlap"], axis=(0, 1)) for metric in group_metrics]
    )

    expected_shape = (len(group_metrics), SF_ALIGNMENT_MODE_COUNT)
    for name, values in (
        ("eigenvalues", sf_eigenvalues_by_animal),
        ("cos2", sf_cos2_by_animal),
        ("overlap", sf_overlap_by_animal),
    ):
        if values.shape != expected_shape:
            raise AssertionError(f"Invalid per-animal {name} shape {values.shape}.")

    finite_cos2 = sf_cos2_by_animal[np.isfinite(sf_cos2_by_animal)]
    finite_overlap = sf_overlap_by_animal[np.isfinite(sf_overlap_by_animal)]
    if finite_cos2.size == 0 or np.any(finite_cos2 < -1e-10):
        raise AssertionError("Figure 1g contains invalid cos2 values.")
    if np.any(finite_cos2 > 1 + 1e-8):
        raise AssertionError("Figure 1g cos2 exceeds one beyond numerical tolerance.")
    if finite_overlap.size == 0 or np.any(finite_overlap < -1e-12):
        raise AssertionError("Figure 1g contains invalid raw overlap values.")
    for name, values in (
        ("eigenvalues", sf_eigenvalues_by_animal),
        ("cos2", sf_cos2_by_animal),
        ("overlap", sf_overlap_by_animal),
    ):
        if np.any(np.sum(np.isfinite(values), axis=0) < 2):
            raise AssertionError(f"Fewer than two animals contribute to a {name} mode.")

    return {
        "eigenvalues_by_animal": sf_eigenvalues_by_animal,
        "cos2_by_animal": sf_cos2_by_animal,
        "overlap_by_animal": sf_overlap_by_animal,
        "x": sf_nanmean(sf_eigenvalues_by_animal, axis=0),
        "cos2_mean": sf_nanmean(sf_cos2_by_animal, axis=0),
        "cos2_sem": sf_nansem(sf_cos2_by_animal, axis=0),
        "overlap_mean": sf_nanmean(sf_overlap_by_animal, axis=0),
        "overlap_sem": sf_nansem(sf_overlap_by_animal, axis=0),
    }

sf_metrics = {
    "FR": sf_analyze_group("FR", FOOD_RESTRICTED_SATED),
    "CTR": sf_analyze_group("CTR", CONTROL_SATED),
}
if len(sf_metrics["FR"]) != 8 or len(sf_metrics["CTR"]) != 6:
    raise AssertionError("Figure 1g expected eight FR and six CTR SATED animals.")

sf_summaries = {
    group_name: sf_summarize_group(group_metrics)
    for group_name, group_metrics in sf_metrics.items()
}


In [ ]:
def sf_prepare_plot_values(summary, value_key, sem_key):
    """Remove invalid/numerical modes before the shared Figure 1e plot helper."""
    x = np.asarray(summary["x"], dtype=float)
    y = np.asarray(summary[value_key], dtype=float)
    sem = np.asarray(summary[sem_key], dtype=float)
    valid = (
        np.isfinite(x)
        & np.isfinite(y)
        & np.isfinite(sem)
        & (x > SF_ALIGNMENT_EIGEN_EPS)
    )
    if not np.any(valid):
        raise ValueError(f"No valid Figure 1g modes remain for {value_key}.")
    return x[valid], y[valid], sem[valid]

sf_fr_cos2_plot = sf_prepare_plot_values(
    sf_summaries["FR"],
    "cos2_mean",
    "cos2_sem",
)
sf_ctr_cos2_plot = sf_prepare_plot_values(
    sf_summaries["CTR"],
    "cos2_mean",
    "cos2_sem",
)
sf_fr_overlap_plot = sf_prepare_plot_values(
    sf_summaries["FR"],
    "overlap_mean",
    "overlap_sem",
)
sf_ctr_overlap_plot = sf_prepare_plot_values(
    sf_summaries["CTR"],
    "overlap_mean",
    "overlap_sem",
)

sf_fig, sf_axes = plot_eigen_cos2_and_overlap_stacked(
    x_fr=sf_fr_cos2_plot[0],
    cos2_fr=sf_fr_cos2_plot[1],
    cos2_sem_fr=sf_fr_cos2_plot[2],
    overlap_fr=sf_fr_overlap_plot[1],
    overlap_sem_fr=sf_fr_overlap_plot[2],
    x_ctr=sf_ctr_cos2_plot[0],
    cos2_ctr=sf_ctr_cos2_plot[1],
    cos2_sem_ctr=sf_ctr_cos2_plot[2],
    overlap_ctr=sf_ctr_overlap_plot[1],
    overlap_sem_ctr=sf_ctr_overlap_plot[2],
    xlabel=r"$\left\langle \lambda_i \right\rangle$",
    cos2_ylabel=r"$\left\langle \cos^2 \theta_i^{\mathrm{SF}} \right\rangle$",
    overlap_ylabel=(
        r"$\left\langle (\nu_i \cdot \Delta \mu_{\mathrm{SF}})^2 \right\rangle$"
    ),
    save_path=None,
)


### ED Figure 2c: Noise orientation and signal overlap — movies

Requires `maleMovieData.mat` in `MOVIE_DATA_DIR` (default: `data/movie/`).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import loadmat

if "CELL_ARRAY" not in globals():
    CELL_ARRAY = loadmat((MOVIE_DATA_DIR / 'maleMovieData.mat'), simplify_cells=True)["all_roiSessData"]

NAT_FR_IDS = [3, 4, 5, 8, 9, 10, 12]
NAT_CTR_IDS = [0, 1, 2, 6, 7, 11]
NAT_GROUP_IDS = {"FR": NAT_FR_IDS, "CTR": NAT_CTR_IDS}

NAT_MOVIE_START = 160
NAT_MOVIE_STOP = 1360
NAT_MOVIE_INDEX = 1       # Python index 1 = natural scenes
NAT_SIGNAL_INDEX = 1      # Python index 1 = inferred spikes
NAT_BIN_FRAMES = 10       # 2 s at 20 Hz, matching the Figure 1e averaging window
NAT_EXPECTED_WINDOWS = NAT_BIN_FRAMES*12
NAT_EXPECTED_PAIRS = NAT_EXPECTED_WINDOWS - 1
NAT_EIGEN_EPS = 1e-8
NAT_SIGNAL_NORM_EPS = 1e-12

NAT_COLORS = {
    "FR": "#a82203",
    "CTR": "#208cc0",
}

NAT_PAPER_STYLE = {
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Helvetica",
        "Arial",
        "Liberation Sans",
        "Nimbus Sans",
        "DejaVu Sans",
    ],
    "font.size": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "lines.linewidth": 1.0,
    "lines.markersize": 3,
    "axes.linewidth": 0.6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "savefig.dpi": 300,
    "savefig.transparent": True,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}

def natural_nanmean(values, axis=0):
    """NaN-safe mean without empty-slice runtime warnings."""
    values = np.asarray(values, dtype=float)
    count = np.sum(np.isfinite(values), axis=axis)
    total = np.nansum(values, axis=axis)
    return np.divide(
        total,
        count,
        out=np.full(np.shape(total), np.nan, dtype=float),
        where=count > 0,
    )

def natural_nansem(values, axis=0):
    """Sample SEM using the finite animal count at each plotted point."""
    values = np.asarray(values, dtype=float)
    count = np.sum(np.isfinite(values), axis=axis)
    mean = natural_nanmean(values, axis=axis)
    centered = np.where(
        np.isfinite(values),
        values - np.expand_dims(mean, axis=axis),
        0.0,
    )
    sum_squares = np.sum(centered**2, axis=axis)
    sem_squared = np.divide(
        sum_squares,
        count * (count - 1),
        out=np.full(np.shape(sum_squares), np.nan, dtype=float),
        where=count > 1,
    )
    return np.sqrt(sem_squared)

def natural_movie_data(animal_id):
    """Return neurons × trials × movie-frames inferred-spike data."""
    data = np.asarray(
        CELL_ARRAY[animal_id][
            :,
            :,
            NAT_MOVIE_START:NAT_MOVIE_STOP,
            NAT_MOVIE_INDEX,
            NAT_SIGNAL_INDEX,
        ],
        dtype=float,
    )
    expected_frames = NAT_MOVIE_STOP - NAT_MOVIE_START
    if data.ndim != 3 or data.shape[2] != expected_frames:
        raise AssertionError(
            f"Animal {animal_id}: expected a neurons × trials × "
            f"{expected_frames} array, found {data.shape}."
        )
    return data

def natural_complete_trial_mask(animal_id):
    """Keep only trials that are finite for every neuron and movie frame."""
    data = natural_movie_data(animal_id)
    return np.all(np.isfinite(data), axis=(0, 2))

def natural_binned_trials(animal_id):
    """Average each complete trial into non-overlapping 2 s movie windows."""
    data = natural_movie_data(animal_id)
    complete_trials = natural_complete_trial_mask(animal_id)

    if np.sum(complete_trials) < 2:
        raise ValueError(f"Animal {animal_id}: fewer than two complete trials.")

    data = data[:, complete_trials, :]
    if not np.all(np.isfinite(data)):
        raise AssertionError(f"Animal {animal_id}: incomplete data remain.")

    n_neurons, n_trials, n_frames = data.shape
    if n_frames % NAT_BIN_FRAMES != 0:
        raise AssertionError(
            f"{n_frames} movie frames are not divisible by {NAT_BIN_FRAMES}."
        )

    n_windows = n_frames // NAT_BIN_FRAMES
    if n_windows != NAT_EXPECTED_WINDOWS:
        raise AssertionError(
            f"Expected {NAT_EXPECTED_WINDOWS} movie windows, found {n_windows}."
        )

    binned = data.reshape(
        n_neurons,
        n_trials,
        n_windows,
        NAT_BIN_FRAMES,
    ).mean(axis=-1)

    return binned, complete_trials

NAT_ALL_IDS = NAT_FR_IDS + NAT_CTR_IDS
NAT_VALID_TRIAL_COUNTS = {
    animal_id: int(np.sum(natural_complete_trial_mask(animal_id)))
    for animal_id in NAT_ALL_IDS
}
NAT_MODE_COUNT = min(
    min(NAT_VALID_TRIAL_COUNTS[animal_id] - 1, natural_movie_data(animal_id).shape[0])
    for animal_id in NAT_ALL_IDS
)

if len(NAT_FR_IDS) != 7 or len(NAT_CTR_IDS) != 6:
    raise AssertionError("Expected 7 FR and 6 CTR natural-movie sessions.")
if NAT_MODE_COUNT != 8:
    raise AssertionError(
        f"Expected a common trial-limited rank of 8 modes, found {NAT_MODE_COUNT}."
    )

def natural_animal_metrics(animal_id, mode_count=NAT_MODE_COUNT):
    """Compute Figure 1e-style metrics for one natural-movie session."""
    binned, complete_trials = natural_binned_trials(animal_id)
    n_neurons, n_trials, n_windows = binned.shape
    n_pairs = n_windows - 1

    if mode_count > min(n_trials - 1, n_neurons):
        raise ValueError(
            f"Animal {animal_id}: {mode_count} modes exceed the covariance rank."
        )
    if n_pairs != NAT_EXPECTED_PAIRS:
        raise AssertionError(
            f"Animal {animal_id}: expected {NAT_EXPECTED_PAIRS} pairs, "
            f"found {n_pairs}."
        )

    mean_response = binned.mean(axis=1)
    eigenvalue_by_pair = np.full((n_pairs, mode_count), np.nan)
    cos2_by_pair = np.full((n_pairs, mode_count), np.nan)
    overlap_by_pair = np.full((n_pairs, mode_count), np.nan)

    for source_window in range(n_pairs):
        trial_response = binned[:, :, source_window].T
        covariance = np.cov(
            trial_response,
            rowvar=False,
            ddof=0,
        )
        covariance = 0.5 * (covariance + covariance.T)

        eigenvalues, eigenvectors = np.linalg.eigh(covariance)
        order = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[order][:mode_count]
        eigenvectors = eigenvectors[:, order][:, :mode_count]

        delta_mu = (
            mean_response[:, source_window + 1]
            - mean_response[:, source_window]
        )
        signal_norm_squared = float(delta_mu @ delta_mu)
        if not np.isfinite(signal_norm_squared):
            continue
        if signal_norm_squared <= NAT_SIGNAL_NORM_EPS:
            continue

        overlap = (eigenvectors.T @ delta_mu) ** 2
        valid_modes = (
            np.isfinite(eigenvalues)
            & (eigenvalues > NAT_EIGEN_EPS)
            & np.isfinite(overlap)
        )

        eigenvalue_by_pair[source_window] = np.where(
            valid_modes,
            eigenvalues,
            np.nan,
        )
        overlap_by_pair[source_window] = np.where(
            valid_modes,
            overlap,
            np.nan,
        )
        cos2_by_pair[source_window] = np.where(
            valid_modes,
            overlap / signal_norm_squared,
            np.nan,
        )

    result = {
        "animal_id": int(animal_id),
        "valid_trial_count": int(np.sum(complete_trials)),
        "eigenvalue_by_pair": eigenvalue_by_pair,
        "cos2_by_pair": cos2_by_pair,
        "overlap_by_pair": overlap_by_pair,
        "eigenvalue": natural_nanmean(eigenvalue_by_pair, axis=0),
        "cos2": natural_nanmean(cos2_by_pair, axis=0),
        "overlap": natural_nanmean(overlap_by_pair, axis=0),
    }
    return result

def natural_group_summary(animal_metrics):
    """Average within animals first, then return group mean and animal SEM."""
    eigenvalue_by_animal = np.stack(
        [metrics["eigenvalue"] for metrics in animal_metrics],
        axis=0,
    )
    cos2_by_animal = np.stack(
        [metrics["cos2"] for metrics in animal_metrics],
        axis=0,
    )
    overlap_by_animal = np.stack(
        [metrics["overlap"] for metrics in animal_metrics],
        axis=0,
    )

    return {
        "eigenvalue_by_animal": eigenvalue_by_animal,
        "cos2_by_animal": cos2_by_animal,
        "overlap_by_animal": overlap_by_animal,
        "eigenvalue_mean": natural_nanmean(eigenvalue_by_animal, axis=0),
        "cos2_mean": natural_nanmean(cos2_by_animal, axis=0),
        "cos2_sem": natural_nansem(cos2_by_animal, axis=0),
        "overlap_mean": natural_nanmean(overlap_by_animal, axis=0),
        "overlap_sem": natural_nansem(overlap_by_animal, axis=0),
    }

natural_metrics = {
    group: [natural_animal_metrics(animal_id) for animal_id in animal_ids]
    for group, animal_ids in NAT_GROUP_IDS.items()
}
natural_summaries = {
    group: natural_group_summary(group_metrics)
    for group, group_metrics in natural_metrics.items()
}


In [ ]:
def natural_mm_to_inch(mm):
    return mm / 25.4

def natural_clean_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(length=2.5, width=0.6)
    return ax

def natural_sorted_curve(summary, mean_key, sem_key):
    """Filter and independently sort one group curve by mean eigenvalue."""
    x = np.asarray(summary["eigenvalue_mean"], dtype=float)
    y = np.asarray(summary[mean_key], dtype=float)
    sem = np.asarray(summary[sem_key], dtype=float)

    valid = (
        np.isfinite(x)
        & (x > NAT_EIGEN_EPS)
        & np.isfinite(y)
        & np.isfinite(sem)
    )
    if not np.all(valid):
        raise AssertionError(
            f"Non-finite plotted values for {mean_key}/{sem_key}."
        )

    order = np.argsort(x[valid])
    x = x[valid][order]
    y = y[valid][order]
    sem = sem[valid][order]

    if np.any(np.diff(x) < 0):
        raise AssertionError("Plotted eigenvalues are not sorted.")

    return x, y, sem

def natural_plot_cos2_and_overlap(
    summaries,
    save_path=Path(
        "final_figures/panels/"
        "natural_movie_eigenvalue_cos2_overlap_FR_CTR.svg"
    ),
):
    """Create the Figure 1e-style stacked natural-movie panel."""

    with plt.rc_context(NAT_PAPER_STYLE):
        fig, axes = plt.subplots(
            2,
            1,
            figsize=(natural_mm_to_inch(100), natural_mm_to_inch(70)),
            sharex=True,
            gridspec_kw={"height_ratios": [1, 1], "hspace": 0.12},
        )
        ax_cos2, ax_overlap = axes

        for group in ("CTR", "FR"):
            color = NAT_COLORS[group]
            x, mean, sem = natural_sorted_curve(
                summaries[group],
                "cos2_mean",
                "cos2_sem",
            )
            ax_cos2.fill_between(
                x,
                mean - sem,
                mean + sem,
                color=color,
                alpha=0.16,
                linewidth=0,
                zorder=1 if group == "CTR" else 2,
            )
            ax_cos2.plot(
                x,
                mean,
                "o-",
                color=color,
                linewidth=1.0,
                markersize=3,
                markerfacecolor=color,
                markeredgecolor="black",
                markeredgewidth=0.3,
                label=group,
                zorder=3 if group == "CTR" else 4,
            )

            x, mean, sem = natural_sorted_curve(
                summaries[group],
                "overlap_mean",
                "overlap_sem",
            )
            ax_overlap.fill_between(
                x,
                mean - sem,
                mean + sem,
                color=color,
                alpha=0.16,
                linewidth=0,
                zorder=1 if group == "CTR" else 2,
            )
            ax_overlap.plot(
                x,
                mean,
                "o-",
                color=color,
                linewidth=1.0,
                markersize=3,
                markerfacecolor=color,
                markeredgecolor="black",
                markeredgewidth=0.3,
                label=group,
                zorder=3 if group == "CTR" else 4,
            )

        ax_cos2.set_ylabel(
            r"$\left\langle \cos^2 \theta_i \right\rangle$"
        )
        ax_overlap.set_xlabel(
            r"$\left\langle \lambda_i \right\rangle$"
        )
        ax_overlap.set_ylabel(
            r"$\left\langle (\nu_i \cdot \Delta \mu)^2 \right\rangle$"
        )

        for ax in axes:
            natural_clean_axis(ax)
            ax.locator_params(axis="x", nbins=4)
            ax.locator_params(axis="y", nbins=4)

        ax_cos2.tick_params(labelbottom=False)
        ax_cos2.legend(
            frameon=False,
            loc="best",
            handlelength=1.5,
            handletextpad=0.4,
            borderaxespad=0.2,
        )

        fig.subplots_adjust(
            left=0.25,
            right=0.97,
            bottom=0.17,
            top=0.98,
            hspace=0.12,
        )

        plt.show()

    return fig, axes, save_path

natural_fig, natural_axes, NAT_SAVE_PATH = natural_plot_cos2_and_overlap(
    natural_summaries, save_path=None
)


## Figure 2 and Extended Data Figure 4: Data analysis

Figure 2a is an illustration. Figure 2d–e and ED 3 are in `mean_field_figures.ipynb`.


### Shared parameter fit for Figure 2b–c and ED Figure 4


In [ ]:
eps = 1e-6

animals = FOOD_RESTRICTED_SATED

animals = FOOD_RESTRICTED_SATED

D_matrix_fr = np.zeros((len(animals), 5, 12))
for i_animal, animal in enumerate(animals):
    data = resort_preprocessing(SATED_DECONV, SATED_ANGLE, SATED_SF,animal)
    stim_response = np.nanmean(data[:, :, :, :, 40:80], axis=-1)   # N x angle x sf x trial
    for sf in range(5):
        for angle in range(12):
            av_response = np.nanmean(stim_response[:, angle, sf, :], axis=-1, keepdims=True)
            D= av_response * (1 - av_response)
            valid = (
                np.isfinite(D)
                & (D > eps)
            )
            Dv = D[valid]
            D_matrix_fr[i_animal, sf, angle] = np.nanmean(D)

animals = CONTROL_SATED

D_matrix_ctr = np.zeros((len(animals), 5, 12))
for i_animal, animal in enumerate(animals):
    data = resort_preprocessing(SATED_DECONV, SATED_ANGLE, SATED_SF,animal)
    stim_response = np.nanmean(data[:, :, :, :, 40:80], axis=-1)   # N x angle x sf x trial
    for sf in range(5):
        for angle in range(12):
            av_response = np.nanmean(stim_response[:, angle, sf, :], axis=-1, keepdims=True)
            D= av_response * (1 - av_response)
            valid = (
                np.isfinite(D)
                & (D > eps)
            )
            Dv = D[valid]
            D_matrix_ctr[i_animal, sf, angle] = np.nanmean(Dv)
D_Ratio = np.nanmean(D_matrix_fr) / np.nanmean(D_matrix_ctr)
print("D_ratio",D_Ratio)


In [ ]:
C_VAL = 1.22*0.71
eps = 1e-6
mode_limit = 10
perturbations = np.full((12, 5, mode_limit, len(CONTROL_SATED)), np.nan)

for i_animal, animal in enumerate(CONTROL_SATED):
    data = remove_neurons(SATED_DECONV, SATED_ANGLE, SATED_SF,animal)
    stim_response = np.nanmean(data[:, :, :, :, 40:80], axis=-1)   # N x angle x sf x trial
    averaged_response = np.nanmean(stim_response, axis=-1)         # N x angle x sf

    for sf in range(5):
        for angle in range(12):
            rho = averaged_response[:, angle, sf]
            D = rho * (1 - rho)
            X = stim_response[:, angle, sf, :]  # N x trials
            valid = (
                np.isfinite(D)
                & (D > eps)
                & np.all(np.isfinite(X), axis=1)
            )

            Xv = X[valid]
            Dv = D[valid]

            if Xv.shape[0] < 2 or Xv.shape[1] < 2:
                continue
            noise_covariance = np.cov(Xv, rowvar=True)
            evals, evecs = np.linalg.eigh(noise_covariance)

            order = np.argsort(evals)[::-1]
            evals = evals[order]
            evecs = evecs[:, order]

            m_diag = (1.0 - D_Ratio* C_VAL) / Dv   # since D_FR == D_CTR

            perturbation_stim = np.sum((evecs ** 2) * m_diag[:, None], axis=0)

            rank = 10
            perturbation_stim = perturbation_stim[:rank]
            evals = evals[:rank]
            perturbations[angle, sf, :,i_animal] = perturbation_stim

average_perturbation_animal = np.nanmean(perturbations, axis=(0, 1))
average_perturbation = np.nanmean(average_perturbation_animal, axis=-1)


In [ ]:
N_MODES = 15
EIGEN_EPS = 1e-8
TIME_WINDOW = (40,80)  # set to (40, 80) for evoked-response covariance only

def _safe_nanmean(arr, axis):
    count = np.sum(np.isfinite(arr), axis=axis)
    total = np.nansum(arr, axis=axis)
    return np.divide(
        total,
        count,
        out=np.full(np.shape(total), np.nan, dtype=float),
        where=count > 0,
    )

def _safe_nansem(arr, axis=0):
    arr = np.asarray(arr, dtype=float)
    count = np.sum(np.isfinite(arr), axis=axis)
    mean = _safe_nanmean(arr, axis=axis)
    centered = arr - np.expand_dims(mean, axis)
    ss = np.nansum(centered**2, axis=axis)
    var = np.divide(
        ss,
        count - 1,
        out=np.full(np.shape(ss), np.nan, dtype=float),
        where=count > 1,
    )
    sd = np.sqrt(var)
    return np.divide(
        sd,
        np.sqrt(count),
        out=np.full(np.shape(sd), np.nan, dtype=float),
        where=count > 0,
    )

def eigenvalues_per_mode(data, n_modes=N_MODES, min_eval=EIGEN_EPS, time_window=TIME_WINDOW, ddof=1):
    if time_window is not None:
        t0, t1 = time_window
        data = data[..., t0:t1]

    trial_average = _safe_nanmean(data, axis=-1)
    _, n_angles, n_sfs, _ = trial_average.shape
    output = np.full((n_modes, n_angles, n_sfs), np.nan)

    for iangle in range(n_angles):
        for jsf in range(n_sfs):
            dat = trial_average[:, iangle, jsf, :]  # neurons x trials
            valid_neurons = np.any(np.isfinite(dat), axis=1)
            dat = dat[valid_neurons]
            if dat.shape[0] == 0:
                continue

            valid_trials = np.all(np.isfinite(dat), axis=0)
            dat = dat[:, valid_trials].T  # trials x neurons
            if dat.shape[0] <= ddof:
                continue

            cov = np.atleast_2d(np.cov(dat, rowvar=False, ddof=ddof))
            try:
                eigenvals = np.linalg.eigvalsh(cov)
            except np.linalg.LinAlgError:
                continue

            eigenvals = np.sort(eigenvals)[::-1]
            eigenvals[eigenvals < min_eval] = np.nan
            n_store = min(n_modes, eigenvals.size)
            output[:n_store, iangle, jsf] = eigenvals[:n_store]

    return _safe_nanmean(output, axis=(1, 2))

fr_evals = np.vstack([
    eigenvalues_per_mode(remove_neurons(SATED_DECONV, SATED_ANGLE, SATED_SF, FR))
    for FR in FOOD_RESTRICTED_SATED
])
ctr_evals = np.vstack([
    eigenvalues_per_mode(remove_neurons(SATED_DECONV, SATED_ANGLE, SATED_SF, CTR))
    for CTR in CONTROL_SATED
])


In [ ]:
r_ratio = 1.22
w_ratio = 0.71
beta_theory = r_ratio * w_ratio   # fixed theoretical slope
c = beta_theory
alpha = average_perturbation[:mode_limit]

lambda_ctr = np.nanmean(ctr_evals[:, :mode_limit], axis=0)
lambda_fr  = np.nanmean(fr_evals[:, :mode_limit], axis=0)

lambda_ctr_sem = np.nanstd(ctr_evals[:, :mode_limit], axis=0, ddof=1) / np.sqrt(ctr_evals.shape[0])
lambda_fr_sem  = np.nanstd(fr_evals[:, :mode_limit], axis=0, ddof=1) / np.sqrt(fr_evals.shape[0])

A = c / np.sqrt(lambda_fr) - c / np.sqrt(lambda_ctr)

valid = (
    np.isfinite(A)
    & np.isfinite(alpha)
    & np.isfinite(lambda_ctr)
    & np.isfinite(lambda_fr)
    & (lambda_ctr > 0)
    & (lambda_fr > 0)
)

sigma_opt = np.nansum(A[valid] * alpha[valid]) / np.nansum(A[valid] ** 2)

print("optimal sigma =", sigma_opt)

sigma = sigma_opt

z_ctr = np.sqrt(lambda_ctr) / sigma
z_fr  = np.sqrt(lambda_fr) / (c * sigma)

z_ctr_sem = 0.5 * lambda_ctr_sem / np.sqrt(lambda_ctr) / sigma
z_fr_sem  = 0.5 * lambda_fr_sem  / np.sqrt(lambda_fr)  / (c * sigma)

valid = (
    np.isfinite(z_ctr)
    & np.isfinite(z_fr)
    & np.isfinite(alpha)
    & (z_ctr > 0)
    & (z_fr > 0)
)

x = 1 / z_ctr[valid]
y = 1 / z_fr[valid]
mode_limit = 10
modes = np.arange(1, mode_limit+1)
m = modes[valid]
alpha = alpha[valid]

xerr = z_ctr_sem[valid] / z_ctr[valid]**2
yerr = z_fr_sem[valid] / z_fr[valid]**2

SIGMA_INFER = sigma_opt


### ED Figure 4a: Inverse linear-response spectrum — perturbation fit


In [ ]:
fig, ax = plt.subplots(figsize=(mm_to_inch(40), mm_to_inch(40)), dpi=500)

ax.errorbar(
    x, y,
    xerr=xerr,
    yerr=yerr,
    fmt="o",
    color=COLORS["Other"],
    ecolor=COLORS["Other"],
    elinewidth=0.6,
    capsize=2,
    capthick=0.6,
    markersize=3,
    markerfacecolor=COLORS["Other"],
    markeredgecolor="black",
    markeredgewidth=0.3,
    label=rf"data"
)

y_modefit = alpha + c * x
order = np.argsort(x)

ax.plot(
    x[order],
    y_modefit[order],
    "o-",
    color=COLORS["Purple"],
    lw=1.0,
    markersize=3,
    markerfacecolor=COLORS["Purple"],
    markeredgecolor="black",
    markeredgewidth=0.3,
    label=rf"perturbed fit",
)

ax.set_xlabel(r"$1/z_i^{\mathrm{CTR}}$")
ax.set_ylabel(r"$1/z_i^{\mathrm{FR}}$")

clean_axis(ax)
ax.legend(frameon=False)
fig.tight_layout(pad=0.3)
plt.show()


### Figure 2b: Measured and FR-like orientation tuning curves


In [ ]:
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
plt.style.use(STYLE_PATH)

def fr_transformation(z_ctr):
    """
    Transform control LR eigenvalues z_i^C into FR-like LR eigenvalues z_i^F.

    The transformation is based on the perturbation analysis of the mixed linear + quadratic model.
    """
    z_fr = 1 / (average_perturbation[:len(z_ctr)] + beta_theory / z_ctr)

    return z_fr

def nansem(x, axis=0):
    x = np.asarray(x, dtype=float)
    n = np.sum(np.isfinite(x), axis=axis)
    sd = np.nanstd(x, axis=axis, ddof=1)

    sem = sd / np.sqrt(n)
    sem = np.where(n > 1, sem, np.nan)

    return sem
def animal_fr_curve(datum, angles, sfs, animal):
    response = get_best_sf_response(datum, angles, sfs, animal)
    return average_aligned_tuning(np.nanmean(response, axis=-1))
def noise_covariance(response):
    residuals = response - np.nanmean(response, axis=-1, keepdims=True)
    residuals = residuals.reshape(response.shape[0], -1)
    return pd.DataFrame(residuals.T).cov().to_numpy()

def make_signal_operator(eigvecs, z, c=1.0):
    """
    Construct the mean-response / signal operator.

    Mean response model:
        mu_centered = c * Gamma x

    where Gamma has eigenvalues z_i.
    """

    signal_eigs = c * z
    return (eigvecs * signal_eigs) @ eigvecs.T

def make_covariance_operator(eigvecs, z, A=1.0, c=1.0):
    """
    Construct covariance from the mixed linear + quadratic model:

        lambda_i = A * c^2 * z_i^2 + B * z_i
    """
    lam = A * c**2 * z**2
    return (eigvecs * lam) @ eigvecs.T
def infer_linear_response_eigs_from_cov(
    cov,
    A=1.0,
    c=1.0,
    eps=1e-12,
):
    """
    Infer latent linear-response eigenvalues z_i from covariance eigenvalues.

        z_i = sqrt(lambda_i)/c
    """
    eigvals, eigvecs = np.linalg.eigh(cov)
    z = np.sqrt(eigvals/(c*SIGMA_INFER))

    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    z = z[:10]
    eigvecs = eigvecs[:, :10]
    lam = eigvals[:10]

    return eigvecs, z, lam

def rescaled_tuning_curve(
    response,
    A_ctr=1.0,
    A_fr=1.0,
    c_ctr=1.0,
    c_fr=0.86,
    r_ratio=1.22,
    return_diagnostics=False,
):

    cov_ctr = noise_covariance(response)

    eigvecs, z_ctr, lambda_ctr = infer_linear_response_eigs_from_cov(
        cov_ctr,
        A=A_ctr,
        c=c_ctr,
    )
    tuning = np.nanmean(response, axis=-1)

    baseline = np.nanmean(tuning, axis=1, keepdims=True)
    centered_tuning = tuning - baseline

    S_ctr = make_signal_operator(
        eigvecs,
        z_ctr,
        c=c_ctr,
    )

    xhat = np.linalg.pinv(S_ctr, rcond=1e-10) @ centered_tuning

    z_fr = fr_transformation(z_ctr)

    S_fr = make_signal_operator(
        eigvecs,
        z_fr,
        c=c_fr,
    )

    predicted_tuning = S_fr @ xhat + baseline

    cov_fr_pred = make_covariance_operator(
        eigvecs,
        z_fr,
        A=A_fr,
        c=c_fr,
    )

    if return_diagnostics:
        diagnostics = {
            "cov_ctr": cov_ctr,
            "lambda_ctr": lambda_ctr,
            "eigvecs": eigvecs,
            "z_ctr": z_ctr,
            "z_fr": z_fr,
            "S_ctr": S_ctr,
            "S_fr": S_fr,
            "xhat": xhat,
            "baseline": baseline,
            "cov_fr_pred": cov_fr_pred,
            "A_ctr": A_ctr,
            "A_fr": A_fr,
            "c_ctr": c_ctr,
            "c_fr": c_fr,
        }

        return predicted_tuning, diagnostics

    return predicted_tuning
def average_aligned_tuning(tuning):
    if tuning.shape[1] == 12:
        tuning = 0.5 * (tuning[:, :6] + tuning[:, 6:])

    aligned = tuning.copy()
    preferred = np.nanargmax(aligned, axis=1)

    for neuron, pref in enumerate(preferred):
        aligned[neuron] = np.roll(aligned[neuron], 3 - pref)

    curve = np.nanmean(aligned, axis=0)
    curve = curve / np.nanmax(curve)

    return np.r_[curve, curve[0]]
def get_best_sf_response(datum, angles, sfs, animal, response_window=(40, 80)):
    data = remove_neurons(datum, angles, sfs, animal)
    data = np.nanmean(data[:, :, :, :, response_window[0]:response_window[1]], axis=-1)

    response = np.full((data.shape[0], data.shape[1], data.shape[3]), np.nan)

    for neuron in range(data.shape[0]):
        best_sf = np.nanargmax(np.nanmean(data[neuron], axis=(0, 2)))
        response[neuron] = data[neuron, :, best_sf, :]

    return response

def animal_control_curves(datum, angles, sfs, animal):
    response = get_best_sf_response(datum, angles, sfs, animal)

    control_curve = average_aligned_tuning(np.nanmean(response, axis=-1))

    rescaled = rescaled_tuning_curve(
        response,
        A_ctr=1.0,
        A_fr=1.0,
        c_ctr=1.0,
        c_fr=0.87,
        r_ratio=1.0,
    )

    rescaled_curve = average_aligned_tuning(rescaled)

    return control_curve, rescaled_curve

def plot_group_tuning_curves(
    error="sem",
    save_path="final_figures/panels/group_tuning_curves.svg",
):
    control_curves = []
    rescaled_curves = []

    for animal in CONTROL_HUNGRY:
        control_curve, rescaled_curve = animal_control_curves(
            HUNGRY_DECONV,
            HUNGRY_ANGLE,
            HUNGRY_SF,
            animal,
        )
        control_curves.append(control_curve)
        rescaled_curves.append(rescaled_curve)

    fr_curves = []

    for animal in FOOD_RESTRICTED_HUNGRY:
        fr_curves.append(
            animal_fr_curve(
                HUNGRY_DECONV,
                HUNGRY_ANGLE,
                HUNGRY_SF,
                animal,
            )
        )

    control_curves = np.asarray(control_curves, dtype=float)
    rescaled_curves = np.asarray(rescaled_curves, dtype=float)
    fr_curves = np.asarray(fr_curves, dtype=float)

    theta = np.r_[np.linspace(-90, 90, 6, endpoint=False), 90]

    def summarize(curves):
        mean = np.nanmean(curves, axis=0)

        if error == "sem":
            err = nansem(curves, axis=0)
        elif error == "sd":
            err = np.nanstd(curves, axis=0, ddof=1)
        elif error == "none":
            err = np.zeros_like(mean)
        else:
            raise ValueError("error must be 'sem', 'sd', or 'none'")

        return mean, err

    mean_control, err_control = summarize(control_curves)
    mean_rescaled, err_rescaled = summarize(rescaled_curves)
    mean_fr, err_fr = summarize(fr_curves)

    ctr_color = COLORS["CTR"]
    fr_color = COLORS["FR"]
    model_color = COLORS["Purple"]

    fig, ax = plt.subplots(figsize=(mm_to_inch(45), mm_to_inch(35)))

    def plot_with_error(x, y, yerr, color, label, zorder):
        ax.errorbar(
            x,
            y,
            yerr=yerr,
            fmt="o-",
            color=color,
            ecolor=color,
            elinewidth=0.6,
            capsize=2,
            capthick=0.6,
            linewidth=1.0,
            markersize=3,
            markerfacecolor=color,
            markeredgecolor="black",
            markeredgewidth=0.3,
            alpha=0.95,
            label=label,
            zorder=zorder,
        )

    plot_with_error(
        theta,
        mean_control,
        err_control,
        ctr_color,
        "CTR",
        zorder=3,
    )

    plot_with_error(
        theta,
        mean_rescaled,
        err_rescaled,
        model_color,
        "FR-like",
        zorder=4,
    )

    plot_with_error(
        theta,
        mean_fr,
        err_fr,
        fr_color,
        "FR",
        zorder=5,
    )

    ax.set_xlabel("Orientation from preferred (°)")
    ax.set_ylabel("Normalized response")

    ax.set_xlim(-95, 95)
    ax.set_xticks([-90, -45, 0, 45, 90])

    ymin = np.nanmin(
        np.concatenate(
            [
                mean_control - err_control,
                mean_rescaled - err_rescaled,
                mean_fr - err_fr,
            ]
        )
    )

    ymax = np.nanmax(
        np.concatenate(
            [
                mean_control + err_control,
                mean_rescaled + err_rescaled,
                mean_fr + err_fr,
            ]
        )
    )

    ax.set_ylim(
        max(0, ymin - 0.04),
        min(1.08, ymax + 0.04),
    )

    ax.locator_params(axis="y", nbins=4)

    clean_axis(ax)

    ax.legend(
        frameon=False,
        loc="upper right",
        handlelength=1.5,
        handletextpad=0.4,
        borderaxespad=0.2,
    )

    fig.tight_layout(pad=0.3)

    if save_path is not None:
        fig.savefig(
            save_path,
            transparent=True,
            bbox_inches="tight",
            pad_inches=0.02,
        )

    plt.show()

    return {
        "theta": theta,
        "control_curves": control_curves,
        "rescaled_curves": rescaled_curves,
        "fr_curves": fr_curves,
        "mean_control": mean_control,
        "mean_rescaled": mean_rescaled,
        "mean_fr": mean_fr,
        "err_control": err_control,
        "err_rescaled": err_rescaled,
        "err_fr": err_fr,
        "fig": fig,
        "ax": ax,
    }

results = plot_group_tuning_curves(
    error="sem",
    save_path=None,
)


### Shared PCA transformation for Figure 2c and ED Figure 4b


In [ ]:
import numpy as np
from sklearn.decomposition import PCA

def PCA_matlab_like(data):
    Tdim=data.shape[0]
    Ndim=data.shape[1]
    pca = PCA(n_components=Ndim)
    if Ndim > Tdim:
        pca = PCA(n_components=Tdim)
    pca.fit(data)
    coeff = pca.components_.T
    score = pca.transform(data)
    eigs = pca.explained_variance_
    if Ndim > Tdim:
        u,s,v = np.linalg.svd(coeff)
        y = u[:,Tdim:]
        coeff = np.hstack([coeff, y])
        eigs = np.append(eigs, np.zeros(Ndim - Tdim))
        score = np.hstack([score, np.zeros((Tdim, Ndim - Tdim))])

    return coeff, score, eigs
n_trial_pcs = 10
animal = 5
start = 50
stop = 80

data = resort_preprocessing(SATED_DECONV, SATED_ANGLE, SATED_SF, animal)
data = data[:, :, :, :, start:stop]

n_neurons, n_angles, n_sfs, n_trials, n_time = data.shape

data_averaged = np.nanmean(data, axis=(2, 3))  # neurons x angles x time
mean_temporal_profile = np.nanmean(data_averaged, axis=1, keepdims=True)
data_centered = data_averaged - mean_temporal_profile
scale = np.sqrt(np.nanvar(data_centered, axis=2, keepdims=True))
scale[scale == 0] = np.nan

X_mean = (data_centered / scale).reshape(n_neurons, -1).T
coeff, score_mean_full, eigs_full = PCA_matlab_like(X_mean)

pca_mean = np.mean(X_mean, axis=0, keepdims=True)
trial_data = (data - mean_temporal_profile[:, :, None, None, :]) / scale[:, :, None, None, :]
trial_data = np.nanmean(trial_data, axis=2)  # neurons x angles x trials x time

trial_points_list = []
trial_angle_index_list = []

for angle_idx in range(n_angles):
    for trial_idx in range(n_trials):
        trial_traj = trial_data[:, angle_idx, trial_idx, :].T
        valid_time = np.all(np.isfinite(trial_traj), axis=1)
        if not np.any(valid_time):
            continue

        trial_score = (trial_traj[valid_time] - pca_mean) @ coeff[:, :n_trial_pcs]
        trial_points_list.append(np.mean(trial_score, axis=0))
        trial_angle_index_list.append(angle_idx)

trial_points = np.asarray(trial_points_list)
trial_angle_index = np.asarray(trial_angle_index_list)
score_mean = score_mean_full[:, :n_trial_pcs]
eigs = eigs_full[:n_trial_pcs]
stimuli_index = np.repeat(np.arange(n_angles), stop - start)

print("trial_points shape:", trial_points.shape)
print("score_mean shape:", score_mean.shape)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

pc_axes = (1, 2)        # 1-based PCs to show, e.g. (1, 2) or (1, 4)
rank = 10              # use the full inferred-rank spectrum for the transform
angle_pair = (0, 1)

c = beta_theory if "beta_theory" in globals() else 1.22 * 0.71
sigma = SIGMA_INFER
chi2_val = 5.991464547  # 95% ellipse, 2 d.o.f.
eps = 1e-12

def finite_rows(x):
    x = np.asarray(x, dtype=float)
    return x[np.all(np.isfinite(x), axis=1)]

def symmetrize(x):
    return 0.5 * (x + x.T)

def group_mode_mean(evals, rank):
    evals = np.asarray(evals, dtype=float)
    if evals.ndim == 1:
        evals = evals[None, :]
    out = np.full(rank, np.nan)
    n = min(rank, evals.shape[1])
    out[:n] = np.nanmean(evals[:, :n], axis=0)
    return out

def angle_points(angle_idx):
    pts = np.asarray(trial_points, dtype=float)[np.asarray(trial_angle_index) == angle_idx, :rank]
    return finite_rows(pts)

def angle_mean(angle_idx):
    if "score_mean" in globals() and "stimuli_index" in globals():
        m = np.nanmean(
            np.asarray(score_mean, dtype=float)[np.asarray(stimuli_index) == angle_idx, :rank],
            axis=0,
        )
        if np.all(np.isfinite(m)):
            return m
    return np.nanmean(angle_points(angle_idx), axis=0)

def angle_cov(angle_idx):
    pts = angle_points(angle_idx)
    if pts.shape[0] < 3:
        raise ValueError(f"Need at least 3 valid trials for angle {angle_idx}.")
    return symmetrize(np.cov(pts, rowvar=False))

def add_ellipse(ax, mean2, cov2, color, fill=True, alpha=0.25, lw=1.0, label=None):
    cov2 = symmetrize(np.asarray(cov2, dtype=float))
    lam, vec = np.linalg.eigh(cov2)
    order = np.argsort(lam)[::-1]
    lam = np.clip(lam[order], 0, None)
    vec = vec[:, order]

    angle = np.degrees(np.arctan2(vec[1, 0], vec[0, 0]))
    width, height = 2 * np.sqrt(lam * chi2_val)

    ellipse = Ellipse(
        mean2,
        width,
        height,
        angle=angle,
        facecolor=color if fill else "none",
        edgecolor=color,
        alpha=alpha if fill else 1.0,
        linewidth=lw,
        fill=fill,
        label=label,
    )
    ax.add_patch(ellipse)
    return ellipse

dims = np.asarray(pc_axes, dtype=int) - 1
all_pts = finite_rows(np.asarray(trial_points, dtype=float)[:, :rank])

cov_ctr_all = symmetrize(np.cov(all_pts, rowvar=False))
direction_eval, U = np.linalg.eigh(cov_ctr_all)
order = np.argsort(direction_eval)[::-1]
U = U[:, order]

alpha = sigma * average_perturbation[:rank]
lambda_ctr_group = group_mode_mean(ctr_evals, rank)
lambda_fr_group = group_mode_mean(fr_evals, rank)

valid_modes = (
    np.isfinite(lambda_ctr_group)
    & (lambda_ctr_group > eps)
    & np.isfinite(lambda_fr_group)
    & (lambda_fr_group > eps)
    & np.isfinite(alpha)
)

z_ctr = np.full(rank, np.nan)
z_ctr[valid_modes] = (
    np.sqrt(lambda_ctr_group[valid_modes]) / sigma
)

z_fr = np.full(rank, np.nan)
z_fr[valid_modes] = 1 / (
    alpha[valid_modes] + c / z_ctr[valid_modes]
)

gain = np.ones(rank)
gain[valid_modes] = (
    c * z_fr[valid_modes] / z_ctr[valid_modes]
)
M = (U * gain) @ U.T

z_fr_actual = np.full(rank, np.nan)
z_fr_actual[valid_modes] = (
    np.sqrt(lambda_fr_group[valid_modes]) / (c * sigma)
)

gain_actual = np.ones(rank)
gain_actual[valid_modes] = (
    c * z_fr_actual[valid_modes] / z_ctr[valid_modes]
)
M_actual = (U * gain_actual) @ U.T

print("z_ctr from group CTR eigenvalues:", np.array2string(z_ctr, precision=3))
print("z_fr from group FR eigenvalues:", np.array2string(z_fr, precision=3))

means = {a: angle_mean(a) for a in angle_pair}
covs = {a: angle_cov(a) for a in angle_pair}

center = 0.5 * (means[angle_pair[0]] + means[angle_pair[1]])
means_fr = {a: center + M @ (means[a] - center) for a in angle_pair}
covs_fr = {a: (M @ covs[a] @ M.T) for a in angle_pair}

means_fr_actual = {a: center + M_actual @ (means[a] - center) for a in angle_pair}
covs_fr_actual = {a: (M_actual @ covs[a] @ M_actual.T) for a in angle_pair}


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Arc

a0, a1 = angle_pair
show_trials = True

groups = {
    "CTR": (means, covs, COLORS["CTR"], "-"),
    "FR reconstruction": (
        means_fr_actual, covs_fr_actual, COLORS["FR"], "-"
    ),
    "FR-like": (means_fr, covs_fr, COLORS["Purple"], "--"),
}

origin = 0.5 * (means[a0] + means[a1])
s_ctr = means[a1] - means[a0]
s_like = means_fr[a1] - means_fr[a0]
s_fr = means_fr_actual[a1] - means_fr_actual[a0]

e1 = s_ctr / np.linalg.norm(s_ctr)
e2 = s_like - e1 * np.dot(e1, s_like)

if np.linalg.norm(e2) < 1e-10 * np.linalg.norm(s_fr):
    e2 = np.eye(len(e1))[:, np.argmin(np.abs(e1))]
    e2 -= e1 * np.dot(e1, e2)

P = np.column_stack((e1, e2 / np.linalg.norm(e2)))

projected = {}
bounds = []

for name, (mu, cov, color, style) in groups.items():
    m = np.array([(mu[a] - origin) @ P for a in angle_pair])
    C = np.array([P.T @ cov[a] @ P for a in angle_pair])
    C = 0.5 * (C + C.transpose(0, 2, 1))
    projected[name] = (m, C, color, style)

    radii = np.sqrt(5.991464547 * np.maximum(
        np.diagonal(C, axis1=1, axis2=2), 0
    ))
    bounds.extend(m - radii)
    bounds.extend(m + radii)

bounds = np.array(bounds)
lower, upper = bounds.min(axis=0), bounds.max(axis=0)
padding = 0.15 * (upper - lower)

def plot_comparison(reference, comparison, filename, show_angle=True):
    fig, ax = plt.subplots(
        figsize=(mm_to_inch(40), mm_to_inch(35)), dpi=500
    )

    if show_trials and reference == "CTR":
        for a in angle_pair:
            points = (angle_points(a) - origin) @ P
            ax.scatter(
                *points.T, s=8, color=COLORS["CTR"],
                alpha=0.15, linewidth=0, zorder=1,
            )

    for name in (reference, comparison):
        m, C, color, style = projected[name]
        filled = name == "CTR"
        hollow = name == "FR-like"

        for i in range(2):
            values, vectors = np.linalg.eigh(C[i])
            width, height = 2 * np.sqrt(
                5.991464547 * np.maximum(values[::-1], 0)
            )
            direction = vectors[:, -1]

            ax.add_patch(Ellipse(
                m[i], width, height,
                angle=np.degrees(np.arctan2(direction[1], direction[0])),
                facecolor=color if filled else "none",
                edgecolor=color,
                alpha=0.20 if filled else 1,
                linestyle=style,
                linewidth=1.0,
                zorder=2,
            ))

        ax.scatter(
            *m.T, s=20,
            facecolors="none" if hollow else color,
            edgecolors=color if hollow else "black",
            linewidths=0.8 if hollow else 0.35,
            label=name, zorder=6 if hollow else 5,
        )

        ax.annotate(
            "", xy=m[1], xytext=m[0],
            arrowprops=dict(
                arrowstyle="->", color=color, linestyle=style,
                linewidth=1, mutation_scale=7, shrinkA=0, shrinkB=0,
            ),
        zorder=4,
        )

    if show_angle:
        m_ref = projected[reference][0]
        m_cmp, _, color, _ = projected[comparison]
        s_ref = m_ref[1] - m_ref[0]
        s_cmp = m_cmp[1] - m_cmp[0]

        start = np.degrees(np.arctan2(s_ref[1], s_ref[0]))
        end = np.degrees(np.arctan2(s_cmp[1], s_cmp[0]))
        rotation = (end - start + 180) % 360 - 180
        radius = 0.4 * np.linalg.norm(s_ref)
        center = m_ref.mean(axis=0)

        ax.add_patch(Arc(
            center, 2 * radius, 2 * radius,
            theta1=min(start, start + rotation),
            theta2=max(start, start + rotation),
            color=color, linewidth=1.2, zorder=7,
        ))

        midpoint = np.radians(start + rotation / 2)
        arc_point = center + radius * np.array([
            np.cos(midpoint), np.sin(midpoint)
        ])

        ax.annotate(
            f"{abs(rotation):.1f}°",
            xy=arc_point, xytext=(8, 18), textcoords="offset points",
            fontsize=7, color=color, ha="center",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=1),
            arrowprops=dict(arrowstyle="-", color=color, linewidth=0.6),
            zorder=8,
        )

    ax.axhline(
        0, color=COLORS["CTR"], linestyle=":",
        linewidth=0.7, alpha=0.7, zorder=0,
    )
    ax.set(
        xlabel="Along CTR signal",
        ylabel="Perpendicular to CTR signal",
        xlim=(lower[0] - padding[0], upper[0] + padding[0]),
        ylim=(lower[1] - padding[1], upper[1] + padding[1]),
    )
    clean_axis(ax)
    ax.legend(frameon=False, loc="upper right", fontsize=6)

    fig.tight_layout(pad=0.3)
    plt.show()
    return fig, ax


### Figure 2c: CTR and FR-like signal/noise geometry


In [ ]:
fig_ctr, ax_ctr = plot_comparison(
    "CTR", "FR-like", "CTR_vs_FR_like"
)


### ED Figure 4b: FR reconstruction and FR-like signal/noise geometry


In [ ]:
fig_like, ax_like = plot_comparison(
    "FR reconstruction", "FR-like", "FR_like_vs_FR_reconstruction", show_angle=False
)


## Extended Data Figure 7: Population SNR dynamics following recovery

Previously food-restricted (FR) and control (CTR) animals during recovery sessions,
after two weeks of ad libitum feeding and restoration of body mass. SNR is calculated
in 50-ms bins relative to stimulus onset. Dashed lines show animal means; shading
shows animal SEM. 


In [ ]:
MIN_EVAL = 1e-10


In [ ]:
from scipy.stats import sem
import pickle

with open((CACHE_DIR / 'results_all_30_recovery_1_bin.pkl'), "rb") as f:
    results = pickle.load(f)
r = results[1]

def summarize_bin_information(overlap, eigvals, min_eval):
    overlap = np.asarray(overlap, dtype=float)
    eigvals = np.asarray(eigvals, dtype=float)

    valid = (
        np.isfinite(overlap)
        & np.isfinite(eigvals)
        & (eigvals > min_eval)
    )

    mode_information = np.full_like(overlap, np.nan)
    np.divide(
        overlap,
        eigvals,
        out=mode_information,
        where=valid,
    )

    bin_information = np.nansum(mode_information, axis=4)
    bin_information[~np.any(valid, axis=4)] = np.nan

    animal_information = np.nanmean(
        bin_information,
        axis=(2, 3, 4),
    )

    group_mean = np.nanmean(animal_information, axis=1)
    group_sem = sem(
        animal_information,
        axis=1,
        nan_policy="omit",
        ddof=1,
    )

    return animal_information, group_mean, group_sem



animal_info_fr, group_mean_fr, group_sem_fr = summarize_bin_information(
    r["fr_overlap_traj"],
    r["fr_eval_traj"],
    min_eval=MIN_EVAL,
)
animal_info_ctr, group_mean_ctr, group_sem_ctr = summarize_bin_information(
    r["ctr_overlap_traj"],
    r["ctr_eval_traj"],
    min_eval=MIN_EVAL,
)

fig, ax = plt.subplots(figsize=(mm_to_inch(45), mm_to_inch(35)))

ax.plot(
    r["time_s"],
    group_mean_fr,
    color=COLORS["FR"],
    lw=1,
    label="FR",
    linestyle="--",
)

ax.plot(
    r["time_s"],
    group_mean_ctr,
    color=COLORS["CTR"],
    lw=1,
    label="CTR",
    linestyle="--",
)
ax.fill_between(
    r["time_s"],
    group_mean_fr - group_sem_fr,
    group_mean_fr + group_sem_fr,
    color=COLORS["FR"],
    alpha=0.3,
)
ax.fill_between(
    r["time_s"],
    group_mean_ctr - group_sem_ctr,
    group_mean_ctr + group_sem_ctr,
    color=COLORS["CTR"],
    alpha=0.3,
)

ax.set_xlabel("Time (s)")
ax.set_ylabel("SNR")

clean_axis(ax)

ax.locator_params(axis="x", nbins=4)
ax.locator_params(axis="y", nbins=4)

ax.legend(
    frameon=False,
    handlelength=1.6,
    handletextpad=0.4,
    borderaxespad=0.2,
)

fig.tight_layout(pad=0.2)

plt.show()


## Figure 3: In-vivo data and predictions

Loads SATED analysis caches. The simulation panel 3c is in
`continuous_attractor_figures.ipynb`.


### Figure 3a: Population SNR and the non-reoriented FR prediction


In [ ]:
import pickle

with open((CACHE_DIR / 'results_all_30_sated_1_bin_geometry.pkl'), "rb") as f:
    results = pickle.load(f)


In [ ]:
FACTOR_SNR_SEP_IDX = 1
ANIMAL_SUBSAMPLE_REPEATS = 1000
ANIMAL_SUBSAMPLE_SEED = 1
MIN_EVAL = 1e-10
FACTOR_SNR_MIN_EVAL = MIN_EVAL

SUBSAMPLE_INTERVAL = (2.5, 97.5)
PLOT_ALTERNATES = True
STATE = "SATED"          # "SATED", "HUNGRY", "RECOVERY"

r_geometry = results[FACTOR_SNR_SEP_IDX]

required_geometry_keys = {
    "time_s",
    "fr_eval_traj",
    "fr_cos2_traj",
    "fr_norm_traj",
    "ctr_eval_traj",
    "ctr_cos2_traj",
    "ctr_norm_traj",
}
missing_geometry_keys = required_geometry_keys.difference(r_geometry)
if missing_geometry_keys:
    raise KeyError(
        "Run the Figure 3 full behavioural pipeline before this cell. "
        f"Missing geometry fields: {sorted(missing_geometry_keys)}"
    )

def factorized_snr_per_animal(evals, norms, cos2, min_eval):
    """
    Return SNR(t, animal), computed as

        sum_k cos²(phi_k) * ||Delta mu||² / lambda_k

    before averaging over angle, spatial frequency, and repeat.
    """
    evals = np.asarray(evals, dtype=float)
    norms = np.asarray(norms, dtype=float)
    cos2 = np.asarray(cos2, dtype=float)

    if evals.ndim != 6:
        raise ValueError(
            "Expected eval/cos² shape "
            "(time, animal, angle, sf, mode, repeat)."
        )
    if cos2.shape != evals.shape:
        raise ValueError(
            f"cos² shape {cos2.shape} does not match eigenvalue shape {evals.shape}."
        )

    expected_norm_shape = evals.shape[:4] + evals.shape[5:]
    if norms.shape != expected_norm_shape:
        raise ValueError(
            f"Norm shape {norms.shape} does not match expected "
            f"{expected_norm_shape}."
        )

    norm_sq_by_mode = norms[..., np.newaxis, :] ** 2
    numerator = cos2 * norm_sq_by_mode
    valid = (
        np.isfinite(numerator)
        & np.isfinite(evals)
        & (evals > min_eval)
    )

    snr_modes = np.full_like(evals, np.nan)
    np.divide(
        numerator,
        evals,
        out=snr_modes,
        where=valid,
    )

    snr_sum = np.nansum(snr_modes, axis=4)

    return np.nanmean(snr_sum, axis=(2, 3, 4))

factor_time_s = np.asarray(r_geometry["time_s"], dtype=float)

factor_fr_evals = np.asarray(r_geometry["fr_eval_traj"], dtype=float)
factor_fr_cos2 = np.asarray(r_geometry["fr_cos2_traj"], dtype=float)
factor_fr_norms = np.asarray(r_geometry["fr_norm_traj"], dtype=float)

factor_ctr_evals = np.asarray(r_geometry["ctr_eval_traj"], dtype=float)
factor_ctr_cos2 = np.asarray(r_geometry["ctr_cos2_traj"], dtype=float)
factor_ctr_norms = np.asarray(r_geometry["ctr_norm_traj"], dtype=float)

fr_snr_by_animal = factorized_snr_per_animal(
    factor_fr_evals,
    factor_fr_norms,
    factor_fr_cos2,
    min_eval=FACTOR_SNR_MIN_EVAL,
)
ctr_snr_by_animal = factorized_snr_per_animal(
    factor_ctr_evals,
    factor_ctr_norms,
    factor_ctr_cos2,
    min_eval=FACTOR_SNR_MIN_EVAL,
)

if fr_snr_by_animal.shape[0] != factor_time_s.size:
    raise ValueError("FR geometry time axis does not match time_s.")
if ctr_snr_by_animal.shape[0] != factor_time_s.size:
    raise ValueError("CTR geometry time axis does not match time_s.")

fr_geometry_signature = factor_fr_evals.shape[:1] + factor_fr_evals.shape[2:]
ctr_geometry_signature = factor_ctr_evals.shape[:1] + factor_ctr_evals.shape[2:]
fr_norm_signature = factor_fr_norms.shape[:1] + factor_fr_norms.shape[2:]
ctr_norm_signature = factor_ctr_norms.shape[:1] + factor_ctr_norms.shape[2:]

if PLOT_ALTERNATES and (
    fr_geometry_signature != ctr_geometry_signature
    or fr_norm_signature != ctr_norm_signature
):
    raise ValueError(
        "FR and CTR non-animal dimensions must match to swap cos² trajectories."
    )

n_fr_animals = factor_fr_evals.shape[1]
n_ctr_animals = factor_ctr_evals.shape[1]
n_matched_animals = min(n_fr_animals, n_ctr_animals)

if n_matched_animals < 1:
    raise ValueError("At least one animal is required in each group.")
if ANIMAL_SUBSAMPLE_REPEATS < 1:
    raise ValueError("ANIMAL_SUBSAMPLE_REPEATS must be at least 1.")

if PLOT_ALTERNATES:
    fr_alternate_by_pair = np.full(
        (n_fr_animals, n_ctr_animals, factor_time_s.size),
        np.nan,
    )
    ctr_alternate_by_pair = np.full_like(fr_alternate_by_pair, np.nan)

    for fr_index in range(n_fr_animals):
        for ctr_index in range(n_ctr_animals):
            fr_alternate_by_pair[fr_index, ctr_index] = (
                factorized_snr_per_animal(
                    factor_fr_evals[:, fr_index:fr_index + 1],
                    factor_fr_norms[:, fr_index:fr_index + 1],
                    factor_ctr_cos2[:, ctr_index:ctr_index + 1],
                    min_eval=FACTOR_SNR_MIN_EVAL,
                )[:, 0]
            )
            ctr_alternate_by_pair[fr_index, ctr_index] = (
                factorized_snr_per_animal(
                    factor_ctr_evals[:, ctr_index:ctr_index + 1],
                    factor_ctr_norms[:, ctr_index:ctr_index + 1],
                    factor_fr_cos2[:, fr_index:fr_index + 1],
                    min_eval=FACTOR_SNR_MIN_EVAL,
                )[:, 0]
            )

curve_names = ["FR", "CTR"]
if PLOT_ALTERNATES:
    curve_names.extend(["FR Non-Reorientated", "CTR-Alternate"])

factorized_snr_resamples = {
    name: np.full(
        (ANIMAL_SUBSAMPLE_REPEATS, factor_time_s.size),
        np.nan,
    )
    for name in curve_names
}

animal_rng = np.random.default_rng(ANIMAL_SUBSAMPLE_SEED)

for resample_index in range(ANIMAL_SUBSAMPLE_REPEATS):
    fr_choice = animal_rng.choice(
        n_fr_animals,
        size=n_matched_animals,
        replace=False,
    )
    ctr_choice = animal_rng.choice(
        n_ctr_animals,
        size=n_matched_animals,
        replace=False,
    )

    factorized_snr_resamples["FR"][resample_index] = np.nanmean(
        fr_snr_by_animal[:, fr_choice],
        axis=1,
    )
    factorized_snr_resamples["CTR"][resample_index] = np.nanmean(
        ctr_snr_by_animal[:, ctr_choice],
        axis=1,
    )

    if PLOT_ALTERNATES:
        factorized_snr_resamples["FR Non-Reorientated"][resample_index] = np.nanmean(
            fr_alternate_by_pair[fr_choice, ctr_choice, :],
            axis=0,
        )
        factorized_snr_resamples["CTR-Alternate"][resample_index] = np.nanmean(
            ctr_alternate_by_pair[fr_choice, ctr_choice, :],
            axis=0,
        )

factorized_snr_summary = {}
for name, samples in factorized_snr_resamples.items():
    interval_low, interval_high = np.nanpercentile(
        samples,
        SUBSAMPLE_INTERVAL,
        axis=0,
    )
    factorized_snr_summary[name] = {
        "mean": np.nanmean(samples, axis=0),
        "interval_low": interval_low,
        "interval_high": interval_high,
    }

factorized_snr_styles = {
    "FR": {
        "color": COLORS["FR"],
        "linestyle": "-",
        "fill_alpha": 0.16,
    },
    "CTR": {
        "color": COLORS["CTR"],
        "linestyle": "-",
        "fill_alpha": 0.16,
    },
    "FR Non-Reorientated": {
        "color": COLORS["FR"],
        "linestyle": "--",
        "fill_alpha": 0.08,
    },
    "CTR-Alternate": {
        "color": COLORS["CTR"],
        "linestyle": "--",
        "fill_alpha": 0.08,
    },
}

fig, ax = plt.subplots(figsize=(mm_to_inch(45), mm_to_inch(35)))

for name in curve_names:
    if name == "CTR-Alternate":
        continue
    summary = factorized_snr_summary[name]
    style = factorized_snr_styles[name]
    if name == "FR Non-Reorientated":
        ax.plot(
            factor_time_s,
            summary["mean"],
            color=style["color"],
            linestyle=':',
            marker="D",
            markersize=1,
            lw=1,
            label=name,
        )
    else:
        ax.plot(
            factor_time_s,
            summary["mean"],
            color=style["color"],
            linestyle=style["linestyle"],
            marker="o",
            markersize=1,
            lw=1,
            label=name,
        )

ax.set_xlabel("Time (s)")
ax.set_ylabel("SNR")

clean_axis(ax)
ax.locator_params(axis="x", nbins=4)
ax.locator_params(axis="y", nbins=4)
ax.legend(
    frameon=False,
    handlelength=1.8,
    handletextpad=0.4,
    borderaxespad=0.2,
)

fig.tight_layout(pad=0.2)

print(
    f"Matched {n_matched_animals} animals per group across "
    f"{ANIMAL_SUBSAMPLE_REPEATS} reproducible subsamples "
    f"(FR: {n_fr_animals}, CTR: {n_ctr_animals})."
)

plt.show()


### Figure 3b: Predicted decision time versus orientation separation



In [ ]:
import pickle

import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.special import ndtr, ndtri

DATA_FILE = (CACHE_DIR / 'results_all_30_sated_1_bin.pkl')

SEPS = np.array([30, 20, 10, 7.5, 5], dtype=float)

OBSERVED = np.array([
    [80.5, 72.5, 68.5, 63.5, 56.5],  # CTR
    [80.0, 72.5, 69.0, 55.5, 52.0],  # FR
])

DT = 0.05
T_MIN = 0.05

N_STARTS = 12
RNG_SEED = 7

BETA_GRID = np.linspace(0.25, 1.50, 101)

with open(DATA_FILE, "rb") as f:
    entry = pickle.load(f)[1]

time_s = np.asarray(entry["time_s"], dtype=float)

bin_snr = np.asarray([
    entry["bin_info_ctr"],
    entry["bin_info_fr"],
], dtype=float)

if time_s.ndim != 1:
    raise ValueError("time_s must be one-dimensional.")

if len(time_s) < 2:
    raise ValueError("Need at least two time bins.")

if not np.all(np.isfinite(time_s)):
    raise ValueError("time_s contains non-finite values.")

if not np.isclose(time_s[0], 0):
    raise ValueError("Expected bins to start at t = 0.")

if not np.allclose(np.diff(time_s), DT):
    raise ValueError("Expected consecutive 50-ms bins.")

if bin_snr.shape != (2, len(time_s)):
    raise ValueError("Unexpected SNR array shape.")

if np.any(bin_snr < 0) or not np.all(np.isfinite(bin_snr)):
    raise ValueError("SNR must be finite and non-negative.")

edges = np.r_[time_s, time_s[-1] + DT]

cumulative_snr = np.column_stack([
    np.zeros(2),
    np.cumsum(bin_snr, axis=1),
])

T_MAX = edges[-1]

snr_scale = (
    np.sin(np.deg2rad(SEPS))
    / np.sin(np.deg2rad(SEPS[0]))
) ** 2

difficulty = (
    np.log(SEPS[0] / SEPS)
    / np.log(SEPS[0] / SEPS[-1])
)

def times_from_parameters(p):
    """
    Four fitted time parameters:

        CTR time at 30 deg
        CTR time at 5 deg
        FR  time at 30 deg
        FR  time at 5 deg

    Intermediate times are linear in log difficulty.
    """
    endpoints = np.asarray(p[:4]).reshape(2, 2)

    return (
        endpoints[:, :1] * (1 - difficulty)
        + endpoints[:, 1:] * difficulty
    )

def information_at_times(times):
    """
    Accumulated neural SNR up to each fitted decision time,
    scaled to each angular separation.
    """
    return np.array([
        snr_scale
        * np.interp(times[g], edges, cumulative_snr[g])

        for g in range(2)
    ])

def accuracy(info, log_scale, beta):
    """
    Psychometric transducer:

        P(correct)
          = Phi[
              0.5 * (I / I_scale)^beta
            ]

    beta = 0.5:
        ideal Gaussian/LDA mapping because I = d'^2.

    beta = 1:
        linear mapping from neural SNR to latent
        behavioural discriminability.
    """
    I_scale = np.exp(log_scale)

    latent_dprime = (
        np.maximum(info, 0) / I_scale
    ) ** beta

    return 100 * ndtr(0.5 * latent_dprime)

def initial_log_scale(info, beta):
    """
    Approximate starting value for I_scale obtained by
    algebraically inverting the psychometric function.
    """
    z = ndtri(OBSERVED / 100)

    required_ratio = (2 * z) ** (1 / beta)

    scale_estimates = info / required_ratio

    valid = (
        np.isfinite(scale_estimates)
        & (scale_estimates > 0)
    )

    if not np.any(valid):
        return 0.0

    return np.log(np.median(scale_estimates[valid]))

TIME_BOUNDS = [(T_MIN, T_MAX)] * 4
SCALE_BOUND = [(np.log(1e-6), np.log(1e6))]

BOUNDS = TIME_BOUNDS + SCALE_BOUND

LOWER, UPPER = np.asarray(BOUNDS).T

CONSTRAINTS = [{
    "type": "ineq",
    "fun": lambda p: p[[1, 3]] - p[[0, 2]],
}]

def fit_fixed_beta(beta, seed=RNG_SEED):

    def predict(p):
        times = times_from_parameters(p)
        info = information_at_times(times)

        return accuracy(
            info,
            log_scale=p[4],
            beta=beta,
        )

    def loss(p):
        prediction = predict(p)

        return np.mean((prediction - OBSERVED) ** 2)

    rng = np.random.default_rng(seed)

    solutions = []

    for start in range(N_STARTS):

        if start == 0:
            endpoints = np.array([
                0.12, 0.65,   # CTR
                0.10, 0.45,   # FR
            ])

            endpoints = np.clip(
                endpoints,
                T_MIN,
                T_MAX,
            )

        else:
            endpoints = np.sort(
                rng.uniform(
                    T_MIN,
                    T_MAX,
                    size=(2, 2),
                ),
                axis=1,
            ).ravel()

        initial_info = information_at_times(
            times_from_parameters(endpoints)
        )

        log_scale0 = initial_log_scale(
            initial_info,
            beta,
        )

        p0 = np.r_[
            endpoints,
            log_scale0,
        ]

        p0 = np.clip(
            p0,
            LOWER,
            UPPER,
        )

        result = minimize(
            loss,
            p0,
            method="SLSQP",
            bounds=BOUNDS,
            constraints=CONSTRAINTS,
            options={
                "maxiter": 2000,
                "ftol": 1e-10,
            },
        )

        feasible = np.all(
            result.x[[1, 3]]
            - result.x[[0, 2]]
            >= -1e-7
        )

        if (
            result.success
            and feasible
            and np.all(np.isfinite(result.x))
            and np.isfinite(result.fun)
        ):
            solutions.append(result)

    if not solutions:
        return None

    best = min(
        solutions,
        key=lambda x: x.fun,
    )

    return {
        "beta": beta,
        "fit": best,
        "mse": best.fun,
        "predicted": predict(best.x),
        "times": times_from_parameters(best.x),
        "info": information_at_times(
            times_from_parameters(best.x)
        ),
        "I_scale": np.exp(best.x[4]),
        "n_converged": len(solutions),
    }

profile = []

for beta in BETA_GRID:

    result = fit_fixed_beta(beta)

    if result is None:
        profile.append(np.nan)
    else:
        profile.append(result["mse"])

profile = np.asarray(profile)

if not np.any(np.isfinite(profile)):
    raise RuntimeError(
        "No beta value produced a converged fit."
    )

best_index = np.nanargmin(profile)
best_beta = BETA_GRID[best_index]

best_result = fit_fixed_beta(best_beta)

times = best_result["times"]

theta_grid = np.geomspace(
    SEPS[-1],
    SEPS[0],
    300,
)

difficulty_grid = (
    np.log(SEPS[0] / theta_grid)
    / np.log(SEPS[0] / SEPS[-1])
)

def time_curve(p, difficulty_values):

    endpoints = np.asarray(
        p[:4]
    ).reshape(2, 2)

    return (
        endpoints[:, :1]
        * (1 - difficulty_values)
        + endpoints[:, 1:]
        * difficulty_values
    )

time_curves = time_curve(
    best_result["fit"].x,
    difficulty_grid,
)


In [ ]:
fig, ax = plt.subplots(
    figsize=(mm_to_inch(45), mm_to_inch(35)),
    dpi=500,
    constrained_layout=True,
)

for g, label in enumerate(["CTR", "FR"]):

    ax.plot(
        theta_grid,
        time_curves[g],
        lw=1.8,
        label=label,
        color=COLORS.get(label, None)
    )

    ax.scatter(
        SEPS,
        times[g],
        s=10,
        color=COLORS.get(label, None),
        zorder=3,
    )

ax.set_xlabel(
    r"$\Delta\theta$ (°)",
)

ax.set_ylabel(
    "Decision time (s)",
)

ax.set_xlim(
    SEPS[0] + 1,
    SEPS[-1] - 0.9,
)

ax.set_xticks(
    SEPS,
    [f"{sep:g}" for sep in SEPS],
)

ax.set_ylim(
    0.12,
    0.6,
)

ax.spines[
    ["top", "right"]
].set_visible(False)

ax.legend(
    frameon=False,
)

plt.show()


#### Figure 3b inset: Accuracy over time for a 10° separation


In [ ]:
sep = 10
j = np.flatnonzero(np.isclose(SEPS, sep))[0]

fit_result = best_result
log_scale = fit_result["fit"].x[4]
beta = fit_result["beta"]

t_end = min(T_MAX, 1.3 * fit_result["times"][:, j].max())
t = np.linspace(0, t_end, 500)

fig, ax = plt.subplots(figsize=(mm_to_inch(30), mm_to_inch(30)), dpi=150, constrained_layout=True)

for g, (label, color) in enumerate([("CTR", "#208cc0"), ("FR", "#a82203")]):
    info = snr_scale[j] * np.interp(t, edges, cumulative_snr[g])
    performance = accuracy(info, log_scale=log_scale, beta=beta)

    td = fit_result["times"][g, j]
    pd = fit_result["predicted"][g, j]

    ax.plot(t, performance, color=color, lw=1, label=label)
    ax.hlines(pd, 0, td, color=color, ls="--", lw=1)
    ax.vlines(td, 45, pd, color=color, ls="--", lw=1)
    ax.scatter(td, pd, color=color, s=10, zorder=3)

ax.set(
    xlabel="Time (s)",
    ylabel="P (%)",
    title=rf"$\Delta\theta = {sep:g}^\circ$",
    xlim=(0, t_end),
    ylim=(48, 70),
    yticks=[50, 60, 70],
)
ax.xaxis.label.set_fontsize(8)
ax.yaxis.label.set_fontsize(8)
ax.tick_params(axis="both", which="major", labelsize=8)
ax.spines[["top", "right"]].set_visible(False)

plt.show()


## Extended Data Figure 8: Fits to the in-vivo data

Uses the SATED/RECOVERY subsample geometry caches in `CACHE_DIR`.
The analytic ED 8d panel is in `mean_field_figures.ipynb`.


### Shared fits for ED Figure 8a–b


In [ ]:
import pickle
from itertools import product

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

MIN_EVAL = 1e-10
NMODES = 10

files = {
    "SATED": (CACHE_DIR / 'results_subsample_30_sated_1_bin_geometry.pkl'),
    "RECOVERY": (CACHE_DIR / 'results_subsample_30_recovery_1_bin_geometry.pkl'),
}
plt.style.use(STYLE_PATH)

def fit_state(filename):
    with open(filename, "rb") as f:
        r = pickle.load(f)[1]

    def average_group(group):
        evals = np.asarray(r[f"{group}_eval_traj"][..., :NMODES, :], float)
        overlaps = np.asarray(r[f"{group}_overlap_traj"][..., :NMODES, :], float)

        keep = (
            np.isfinite(evals)
            & np.isfinite(overlaps)
            & (evals >= MIN_EVAL)
            & (overlaps >= 0)
        )

        evals = np.nanmean(
            np.where(keep, evals, np.nan),
            axis=(1, 2, 3, 5),
        )
        overlaps = np.nanmean(
            np.where(keep, overlaps, np.nan),
            axis=(1, 2, 3, 5),
        )

        return evals, overlaps

    fr_eval, fr_overlap = average_group("fr")
    ctr_eval, ctr_overlap = average_group("ctr")

    with np.errstate(divide="ignore", invalid="ignore"):
        ar = fr_overlap / ctr_overlap
        lr = fr_eval / ctr_eval
        a_drive = np.sqrt(ctr_overlap)

    valid = (
        np.isfinite(ar)
        & np.isfinite(lr)
        & np.isfinite(a_drive)
        & (ar >= 0)
    )

    nm = ar.shape[1]

    def predict(params):
        gamma, delta = params[:2]
        p = params[2:]
        noise_drive = a_drive * p[None, :]

        return (
            gamma * np.sqrt(np.clip(ar, 0, None))
            + delta * noise_drive * ar
        ) / (1 + noise_drive)

    def residuals(params):
        noise_drive = a_drive * params[2:][None, :]

        if np.any((1 + noise_drive)[valid] <= 0):
            return np.full(valid.sum(), 1e6)

        prediction = predict(params)

        if not np.all(np.isfinite(prediction[valid])):
            return np.full(valid.sum(), 1e6)

        return prediction[valid] - lr[valid]

    lower = np.r_[1e-8, 1e-8, [-20.0] * nm]
    upper = np.r_[np.inf, np.inf, [20.0] * nm]

    fits = []

    for gamma0, delta0, p0 in product(
        (0.5, 1.0, 1.5),
        (0.5, 1.0, 1.5),
        (-3.0, -1.0, 0.1),
    ):
        initial = np.r_[
            gamma0,
            delta0,
            np.full(nm, p0),
        ]

        fits.append(
            least_squares(
                residuals,
                initial,
                bounds=(lower, upper),
                loss="linear",
                max_nfev=50000,
            )
        )

    fit = min(
        (fit for fit in fits if fit.success),
        key=lambda fit: np.sum(residuals(fit.x) ** 2),
    )

    gamma, delta = fit.x[:2]
    prediction = predict(fit.x)

    ss_res = np.sum((lr[valid] - prediction[valid]) ** 2)
    ss_tot = np.sum((lr[valid] - np.mean(lr[valid])) ** 2)
    r2 = 1 - ss_res / ss_tot

    g_r = 1 / np.sqrt(delta)
    c_inferred = np.sqrt(delta) / gamma

    return (
    c_inferred,
    g_r,
    r2,
    lr[valid],
    prediction[valid],
)

fit_results = {
    state: fit_state(filename)
    for state, filename in files.items()
}

C_inferred = {
    state: values[0]
    for state, values in fit_results.items()
}

for state, (c_value, g_r_value, r2_value, _, _) in fit_results.items():
    print(
        f"{state}: C = {c_value:.4f}, "
        f"R² = {r2_value:.4f}"
    )


### ED Figure 8a: Inferred gain factor during food restriction and recovery


In [ ]:
states = list(C_inferred)
values = [C_inferred[state] for state in states]

x = np.arange(len(states))
bar_colors = [COLORS["Other"], COLORS["compare"]]

fig, ax = plt.subplots(
    figsize=(mm_to_inch(40), mm_to_inch(35)),
    dpi=500,
)

ax.bar(
    x,
    values,
    width=0.62,
    color=bar_colors,
    edgecolor="black",
    linewidth=0.4,
)

ax.axhline(
    1,
    color="0.4",
    linestyle="--",
    linewidth=0.6,
    zorder=0,
)

ax.axhline(
    0.866,
    color="0.4",
    linestyle="--",
    linewidth=0.6,
    zorder=0,
)

ax.set_xticks(x)
ax.set_xticklabels(["Sated", "Recovery"])
ax.set_ylabel(r"Inferred $C_R$")

ax.set_ylim(0.4, max(1.05, np.max(values)) * 1.12)
ax.set_xlim(-0.6, len(states) - 0.4)

clean_axis(ax)
fig.tight_layout(pad=0.3)

plt.show()


### ED Figure 8b: Predicted versus observed covariance ratios


In [ ]:
all_observed = np.concatenate([
    result[3] for result in fit_results.values()
])
all_predicted = np.concatenate([
    result[4] for result in fit_results.values()
])

limit_min = min(all_observed.min(), all_predicted.min())
limit_max = max(all_observed.max(), all_predicted.max())
padding = 0.04 * (limit_max - limit_min)
limits = [limit_min - padding,2]

state_colors = {
    "SATED": COLORS["Other"],
    "RECOVERY": COLORS["compare"],
}

fig, ax = plt.subplots(
    figsize=(mm_to_inch(45), mm_to_inch(40)),
    dpi=500,
)

for state, result in fit_results.items():
    _, _, r2_value, observed, predicted = result

    ax.scatter(
        observed,
        predicted,
        s=5,
        color=state_colors[state],
        alpha=0.55,
        edgecolors="none",
        label=rf"{state.title()}, $R^2={r2_value:.2f}$",
    )

ax.plot(
    limits,
    limits,
    linestyle="--",
    linewidth=0.6,
    color="black",
)

ax.set_xlim(limits)
ax.set_ylim(limits)
ax.set_aspect("equal", adjustable="box")

ax.set_xlabel(r"Observed $LR$")
ax.set_ylabel(r"Predicted $LR$")

clean_axis(ax)

ax.legend(
    frameon=False,
    handlelength=1,
    borderaxespad=0.2,
    markerscale=1.3,
)

fig.tight_layout(pad=0.3)

plt.show()


### ED Figure 8c: Time course of the inferred recurrent gain


In [ ]:
import pickle
from itertools import product

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

MIN_EVAL = 1e-10
NMODES =10
C_FIXED = 0.909
USE_INITIAL_PLATEAU = False

with open((CACHE_DIR / 'results_subsample_30_sated_1_bin_geometry.pkl'), "rb") as f:
    r = pickle.load(f)[1]

def average_group(group):
    evals = np.asarray(r[f"{group}_eval_traj"][..., :NMODES, :], float)
    overlaps = np.asarray(r[f"{group}_overlap_traj"][..., :NMODES, :], float)

    keep = (
        np.isfinite(evals)
        & np.isfinite(overlaps)
        & (evals >= MIN_EVAL)
        & (overlaps >= 0)
    )

    evals = np.nanmean(np.where(keep, evals, np.nan), axis=(1, 2, 3, 5))
    overlaps = np.nanmean(np.where(keep, overlaps, np.nan), axis=(1, 2, 3, 5))

    return evals, overlaps

fr_eval, fr_overlap = average_group("fr")
ctr_eval, ctr_overlap = average_group("ctr")

with np.errstate(divide="ignore", invalid="ignore"):
    ar = fr_overlap / ctr_overlap
    lr = fr_eval / ctr_eval
    a_drive = np.sqrt(ctr_overlap)

valid = (
    np.isfinite(ar)
    & np.isfinite(lr)
    & np.isfinite(a_drive)
    & (ar >= 0)
)

time_s = np.asarray(r["time_s"], float)
time_s = time_s - time_s[0]
nm = ar.shape[1]

def predict(params):
    g_inf, amplitude, tau = np.exp(params[:3])
    if USE_INITIAL_PLATEAU:
        t_flat = params[3]
        p = params[4:]
    else:
        t_flat = 0.0
        p = params[3:]

    decay_time = np.clip(time_s - t_flat, 0, None)
    g_r = g_inf + amplitude * np.exp(-decay_time / tau)
    noise_drive = a_drive * p[None, :]

    prediction = (
        np.sqrt(np.clip(ar, 0, None)) / (C_FIXED * g_r[:, None])
        + noise_drive * ar / g_r[:, None] ** 2
    ) / (1 + noise_drive)

    return prediction, g_r

def residuals(params):
    p = params[4:] if USE_INITIAL_PLATEAU else params[3:]
    noise_drive = a_drive * p[None, :]

    if np.any((1 + noise_drive)[valid] <= 0):
        return np.full(valid.sum(), 1e6)

    prediction, _ = predict(params)

    if not np.all(np.isfinite(prediction[valid])):
        return np.full(valid.sum(), 1e6)

    return prediction[valid] - lr[valid]

if USE_INITIAL_PLATEAU:
    lower = np.r_[[-np.inf] * 3, 0.0, [-20.0] * nm]
    upper = np.r_[[np.inf] * 3, time_s[-2], [20.0] * nm]
else:
    lower = np.r_[[-np.inf] * 3, [-20.0] * nm]
    upper = np.r_[[np.inf] * 3, [20.0] * nm]

fits = []
t_flat_starts = (0.0, 0.15, 0.4) if USE_INITIAL_PLATEAU else (0.0,)

for g0, tau0, t_flat0, p0 in product(
    (0.6, 1.0, 1.4),
    (0.1, 0.3, 0.8),
    t_flat_starts,
    (-3.0, -1.0, 0.0),
):
    initial = np.r_[np.log([g0, 0.5, tau0])]
    if USE_INITIAL_PLATEAU:
        initial = np.r_[initial, t_flat0]
    initial = np.r_[initial, np.full(nm, p0)]

    fits.append(
        least_squares(
            residuals,
            initial,
            bounds=(lower, upper),
            loss="linear",
            max_nfev=50000,
        )
    )

fit_gR = min(
    (fit for fit in fits if fit.success),
    key=lambda fit: np.sum(residuals(fit.x) ** 2),
)

g_inf, amplitude, tau_s = np.exp(fit_gR.x[:3])
if USE_INITIAL_PLATEAU:
    t_flat_s = fit_gR.x[3]
    p_inferred_gR = fit_gR.x[4:]
else:
    t_flat_s = 0.0
    p_inferred_gR = fit_gR.x[3:]
lr_fitted_gR, gR_inferred = predict(fit_gR.x)

ss_res = np.sum((lr[valid] - lr_fitted_gR[valid]) ** 2)
ss_tot = np.sum((lr[valid] - np.mean(lr[valid])) ** 2)
r2_gR = 1 - ss_res / ss_tot

print(f"Fixed C = {C_FIXED:.3f}")
print(f"g_inf = {g_inf:.4f}")
print(f"Amplitude = {amplitude:.4f}")
print(f"Initial plateau = {t_flat_s:.4f} s")
print(f"tau = {tau_s:.4f} s")
print(f"R² = {r2_gR:.4f}")

fig, ax = plt.subplots(
    figsize=(mm_to_inch(30), mm_to_inch(30)),
    dpi=500,
)

ax.plot(
    time_s,
    gR_inferred,
    marker="o",
    markersize=2.5,
    linewidth=0.8,
    color=COLORS["Theory"],
    markeredgewidth=0,
)

ax.set_ylabel(r"$\gamma_R(t)$")
ax.set_xlim(time_s[0], time_s[-1])

clean_axis(ax)
fig.tight_layout(pad=0.3)

plt.show()


## Optional: regenerate geometry caches

Run **Setup** and the recovery-data loader under **ED Figure 1a** first.
Set `RUN_GEOMETRY_CACHE_GENERATION = True` to generate the inputs for Figure 3a
and ED Figure 8a–c from the raw data.

Files are saved under `CACHE_DIR / "regenerated_geometry"`; existing files are
not overwritten.


In [ ]:
RUN_GEOMETRY_CACHE_GENERATION = False
GEOMETRY_CACHE_JOBS = [
    ("SATED", False),     # Figure 3a: all neurons
    ("SATED", True),      # ED Figure 8a–c: subsampling
    ("RECOVERY", True),   # ED Figure 8a–b: subsampling
]
GEOMETRY_CACHE_OUTPUT_DIR = CACHE_DIR / "regenerated_geometry"


In [ ]:
def regenerate_geometry_cache(state="SATED", use_subsampling=False, output_dir="regenerated_geometry"):
    """Compute and save geometry metrics for one state."""
    from pathlib import Path

    import numpy as np
    from scipy.special import erf

    STATE = state          # "SATED", "HUNGRY", "RECOVERY"
    SEPS = [1]         # 1=30°, 2=60°, 3=90°

    TIME_WINDOW = (40, 80)   # frames; 40 is stimulus onset if time = frame/20 - 2
    BIN_WIDTH = 1            # non-overlapping bin width in frames
    FRAME_RATE = 20

    MIN_EVAL = 1e-14
    TOTAL_K = 13             # number of covariance modes to use
    ALPHA = 1.0              # behavioural calibration scale

    USE_SUBSAMPLING = use_subsampling   # True: equal neuron subsample, False: all neurons
    SUBSAMPLE_N_MAX = 50
    SUBSAMPLE_REPEATS = 50
    ALL_NEURONS_REPEATS = 1

    RNG_SEED = 1
    N_SFS = 5

    STATE_MAP_OVERLAP = {
        "SATED":    (SATED_DECONV, SATED_ANGLE, SATED_SF, FOOD_RESTRICTED_SATED, CONTROL_SATED),
        "HUNGRY":   (HUNGRY_DECONV, HUNGRY_ANGLE, HUNGRY_SF, FOOD_RESTRICTED_HUNGRY, CONTROL_HUNGRY),
        "RECOVERY": (RECOVERY_DECONV, RECOVERY_ANGLE, RECOVERY_SF, FOOD_RESTRICTED_RECOVERY, CONTROL_RECOVERY),
    }

    def performance_conversion(cum_info, alpha=1.0):
        """Convert cumulative information to 2AFC accuracy."""
        dprime = np.sqrt(np.maximum(alpha * cum_info, 0))
        return 0.5 * (1 + erf(dprime / (2 * np.sqrt(2))))

    def local_overlap_windows(time_window=(40, 80), bin_width=5, step=1):
        """Return (center, start, stop) bins; stop is exclusive."""
        t0, t1 = map(int, time_window)

        if bin_width < 1:
            raise ValueError("bin_width must be >= 1")
        if step < 1:
            raise ValueError("step must be >= 1")
        if t1 <= t0:
            raise ValueError("time_window must satisfy start < stop")

        windows = []
        for start in range(t0, t1 - bin_width + 1, step):
            stop = start + bin_width
            center = start + bin_width // 2
            windows.append((center, start, stop))

        return windows

    def load_data_by_id(datum, angles, sfs, animal_ids):
        data_by_id = {}
        n_by_id = {}

        for aid in animal_ids:
            data = resort_preprocessing(datum, angles, sfs, aid)
            data = np.asarray(data, dtype=float)

            if data.ndim == 5 and data.shape[0] > 0:
                data_by_id[aid] = data
                n_by_id[aid] = int(data.shape[0])

        return data_by_id, n_by_id

    def compute_signal_vectors(response, t0=40, t1=80):
        """Mean-response differences by neuron, angle pair, and spatial frequency."""
        resp = np.nanmean(response[..., t0:t1], axis=-1)
        mean_resp = np.nanmean(resp, axis=-1)

        dm = mean_resp[:, :, np.newaxis, :] - mean_resp[:, np.newaxis, :, :]

        return dm

    def compute_preferred_sf(response):
        """Preferred spatial frequency per neuron."""
        return np.argmax(np.nanmean(response[:, :, :, :, 40:80], axis=(1, 3, 4)), axis=1)

    def compute_noise_metrics_all(
        response,
        signal_vectors,
        total_k=13,
        t0=40,
        t1=80,
        sample_n=None,
        repeats=50,
        rng_seed=1,
    ):
        """Covariance eigenvalues and squared signal projections for one bin."""
        resp = np.nanmean(response[..., t0:t1], axis=-1)
        n_rois, n_angles, _, n_trials = resp.shape

        if sample_n is None:
            sample_n = n_rois

        sample_n = min(sample_n, n_rois)

        if total_k is None:
            kmax = sample_n
        else:
            kmax = int(total_k)

        n_sfs = N_SFS

        dot_prod = np.full((n_angles, n_angles - 1, n_sfs, kmax, repeats), np.nan)
        top_evals = np.full((n_angles, n_sfs, kmax, repeats), np.nan)
        norm = np.full((n_angles, n_angles - 1, n_sfs, repeats), np.nan)

        sfs_pref = compute_preferred_sf(response)

        for r in range(repeats):
            rng = np.random.default_rng(rng_seed + r)

            idx = rng.choice(n_rois, sample_n, replace=False)

            resp_small = resp[idx]
            sig_vec = signal_vectors[idx]
            sfs_small = sfs_pref[idx]

            for sf in range(n_sfs):
                for ia in range(n_angles):

                    if N_SFS == 1:
                        dat = np.zeros((resp_small.shape[0], resp_small.shape[3]))
                        for i in range(resp_small.shape[0]):
                            best_sf = sfs_small[i]
                            dat[i, :] = resp_small[i, ia, best_sf, :]
                    else:
                        dat = resp_small[:, ia, sf, :]

                    dat_T = dat.T
                    covariance_matrix = np.cov(dat_T, rowvar=False, ddof=0)

                    try:
                        evals, evecs = np.linalg.eigh(covariance_matrix)
                    except np.linalg.LinAlgError:
                        continue

                    order = np.argsort(evals)[::-1]
                    sorted_evals = evals[order]
                    sorted_evecs = evecs[:, order]

                    n_available = min(kmax, sorted_evals.size)
                    top_evals[ia, sf, :n_available, r] = sorted_evals[:n_available]

                    others = (ia + np.arange(1, n_angles)) % n_angles

                    for j, ja in enumerate(others):

                        if N_SFS == 1:
                            sv = sig_vec[np.arange(resp_small.shape[0]), ia, ja, sfs_small]
                        else:
                            sv = sig_vec[:, ia, ja, sf]

                        norm[ia, j, sf, r] = np.linalg.norm(sv)

                        for k in range(n_available):
                            pc = sorted_evecs[:, k]
                            dot_prod[ia, j, sf, k, r] = np.dot(sv, pc) ** 2

        return {
            "dot_prod": dot_prod,
            "top_evals": top_evals,
            "norm": norm,
        }

    def compute_metrics_for_bin(
        data,
        start,
        stop,
        total_k=13,
        sample_n=None,
        repeats=50,
        rng_seed=1,
    ):
        signal_vectors = compute_signal_vectors(data, t0=start, t1=stop)

        metrics = compute_noise_metrics_all(
            data,
            signal_vectors,
            total_k=total_k,
            t0=start,
            t1=stop,
            sample_n=sample_n,
            repeats=repeats,
            rng_seed=rng_seed,
        )

        return metrics

    def metrics_to_bin_outputs(metrics, sep_idx, min_eval=1e-8):
        evals = np.asarray(metrics["top_evals"], dtype=float)
        overlaps = np.asarray(metrics["dot_prod"][:, sep_idx - 1, :, :, :], dtype=float)
        norms = np.asarray(metrics["norm"][:, sep_idx - 1, :, :], dtype=float)

        norm_sq_by_mode = norms[:, :, np.newaxis, :] ** 2
        valid_cos2 = (
            np.isfinite(overlaps)
            & np.isfinite(norm_sq_by_mode)
            & (norm_sq_by_mode > 0)
        )
        cos2 = np.full_like(overlaps, np.nan)
        np.divide(
            overlaps,
            norm_sq_by_mode,
            out=cos2,
            where=valid_cos2,
        )

        evals_for_snr = np.where(evals < min_eval, np.nan, evals)

        with np.errstate(divide="ignore", invalid="ignore"):
            snr_modes = overlaps / evals_for_snr

        snr_total = np.nansum(snr_modes, axis=2)
        bin_info = np.nanmean(snr_total)

        return bin_info, evals, overlaps, cos2, norms

    def compute_condition_bin_info(
        data_by_id,
        animal_ids,
        sep_idx,
        windows,
        total_k=13,
        sample_n=None,
        repeats=50,
        min_eval=1e-8,
        rng_seed=1,
    ):
        info_bins = []
        time_frames = []
        eval_traj = []
        overlap_traj = []
        cos2_traj = []
        norm_traj = []

        for endpoint, start, stop in windows:
            animal_infos = []
            animal_evals = []
            animal_overlaps = []
            animal_cos2 = []
            animal_norms = []

            for aid in animal_ids:
                if aid not in data_by_id:
                    continue

                metrics = compute_metrics_for_bin(
                    data_by_id[aid],
                    start=start,
                    stop=stop,
                    total_k=total_k,
                    sample_n=sample_n,
                    repeats=repeats,
                    rng_seed=rng_seed,
                )

                if metrics is None:
                    continue

                info, evals, overlaps, cos2, norms = metrics_to_bin_outputs(
                    metrics,
                    sep_idx=sep_idx,
                    min_eval=min_eval,
                )

                if np.isfinite(info):
                    animal_infos.append(info)
                    animal_evals.append(evals)
                    animal_overlaps.append(overlaps)
                    animal_cos2.append(cos2)
                    animal_norms.append(norms)

            if len(animal_infos) > 0:
                info_bins.append(np.nanmean(animal_infos))
                time_frames.append(endpoint)
                eval_traj.append(np.stack(animal_evals, axis=0))
                overlap_traj.append(np.stack(animal_overlaps, axis=0))
                cos2_traj.append(np.stack(animal_cos2, axis=0))
                norm_traj.append(np.stack(animal_norms, axis=0))

        return (
            np.asarray(time_frames),
            np.asarray(info_bins),
            np.asarray(eval_traj),
            np.asarray(overlap_traj),
            np.asarray(cos2_traj),
            np.asarray(norm_traj),
        )

    datum, angles, sfs, fr_ids, ctr_ids = STATE_MAP_OVERLAP[STATE]

    all_ids = list(fr_ids) + list(ctr_ids)
    data_by_id, n_by_id = load_data_by_id(datum, angles, sfs, all_ids)

    if len(n_by_id) == 0:
        raise ValueError("No valid animals found.")

    if USE_SUBSAMPLING:
        RUN_SAMPLE_N = min(min(n_by_id.values()), SUBSAMPLE_N_MAX)
        RUN_REPEATS = SUBSAMPLE_REPEATS
    else:
        RUN_SAMPLE_N = None      # None means use all neurons for each animal
        RUN_REPEATS = ALL_NEURONS_REPEATS

    windows = local_overlap_windows(
        time_window=TIME_WINDOW,
        bin_width=BIN_WIDTH)

    results = {}

    for sep_idx in SEPS:
        (
            frames_fr,
            bin_info_fr,
            fr_eval_traj,
            fr_overlap_traj,
            fr_cos2_traj,
            fr_norm_traj,
        ) = compute_condition_bin_info(
            data_by_id,
            fr_ids,
            sep_idx=sep_idx,
            windows=windows,
            total_k=TOTAL_K,
            sample_n=RUN_SAMPLE_N,
            repeats=RUN_REPEATS,
            min_eval=MIN_EVAL,
            rng_seed=RNG_SEED,
        )

        (
            frames_ctr,
            bin_info_ctr,
            ctr_eval_traj,
            ctr_overlap_traj,
            ctr_cos2_traj,
            ctr_norm_traj,
        ) = compute_condition_bin_info(
            data_by_id,
            ctr_ids,
            sep_idx=sep_idx,
            windows=windows,
            total_k=TOTAL_K,
            sample_n=RUN_SAMPLE_N,
            repeats=RUN_REPEATS,
            min_eval=MIN_EVAL,
            rng_seed=RNG_SEED,
        )

        if len(frames_fr) == 0 or len(frames_ctr) == 0:
            raise ValueError(f"No valid bins for sep_idx={sep_idx}.")

        # Align times to stimulus onset.
        time_s = frames_fr / FRAME_RATE - 2

        if not np.array_equal(frames_fr, frames_ctr):
            bin_info_ctr = np.interp(frames_fr, frames_ctr, bin_info_ctr)

        # Accumulate information across time bins.
        cum_info_fr = np.cumsum(bin_info_fr)
        cum_info_ctr = np.cumsum(bin_info_ctr)

        perf_fr = performance_conversion(cum_info_fr, alpha=ALPHA)
        perf_ctr = performance_conversion(cum_info_ctr, alpha=ALPHA)

        results[sep_idx] = {
            "time_s": time_s,
            "bin_info_fr": bin_info_fr,
            "bin_info_ctr": bin_info_ctr,
            "fr_eval_traj": fr_eval_traj,
            "fr_overlap_traj": fr_overlap_traj,
            "fr_cos2_traj": fr_cos2_traj,
            "fr_norm_traj": fr_norm_traj,
            "ctr_eval_traj": ctr_eval_traj,
            "ctr_overlap_traj": ctr_overlap_traj,
            "ctr_cos2_traj": ctr_cos2_traj,
            "ctr_norm_traj": ctr_norm_traj,
            "cum_info_fr": cum_info_fr,
            "cum_info_ctr": cum_info_ctr,
            "perf_fr": perf_fr,
            "perf_ctr": perf_ctr,
            }

    import pickle

    sampling_label = "subsample" if USE_SUBSAMPLING else "all"
    angle_label = "-".join(str(30 * sep_idx) for sep_idx in SEPS)
    window_label = "1_bin" if BIN_WIDTH == 1 else f"overlapping_{BIN_WIDTH}"
    results_path = (
        f"results_{sampling_label}_{angle_label}_{STATE.lower()}_"
        f"{window_label}_geometry.pkl"
    )

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    results_path = output_dir / results_path

    with open(results_path, "xb") as f:
        pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Saved geometry results to {results_path}")

    return results_path


In [ ]:
if RUN_GEOMETRY_CACHE_GENERATION:
    for geometry_state, geometry_subsampling in GEOMETRY_CACHE_JOBS:
        regenerate_geometry_cache(
            state=geometry_state,
            use_subsampling=geometry_subsampling,
            output_dir=GEOMETRY_CACHE_OUTPUT_DIR,
        )
